# Experiments

Dieses Notebook enth?lt die systematische Fidelity/Sparsity-Evaluation f?r GNNExplainer und Integrated Gradients.

- Fidelity wird getrennt f?r `H` und `C` aggregiert.
- Standardm??ig wird nur der **erste Graph pro `compound`** im gew?hlten Scope verwendet.
- Scope kann zwischen `test_split` und `full_dataset` umgeschaltet werden.


In [2]:
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output


def _resolve_project_root() -> Path:
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "gnn4nmr",
        cwd.parent / "gnn4nmr",
    ]
    for candidate in candidates:
        if (candidate / "scripts").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError(
        f"Could not resolve project root from cwd={cwd}. Expected a folder containing scripts/ and notebooks/."
    )


PROJECT_ROOT = _resolve_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [str(PROJECT_ROOT), str(SCRIPTS_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"cwd: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Models dir: {MODELS_DIR}")


cwd: /Users/sophiaberg/gnn4nmr-7/notebooks
Project root: /Users/sophiaberg/gnn4nmr-7
Notebook dir: /Users/sophiaberg/gnn4nmr-7/notebooks
Data dir: /Users/sophiaberg/gnn4nmr-7/data
Models dir: /Users/sophiaberg/gnn4nmr-7/models


In [3]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# Make this cell runnable on its own (even after kernel restart)
if "NOTEBOOK_DIR" not in globals():
    NOTEBOOK_DIR = Path.cwd()
if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
if "MODELS_DIR" not in globals():
    MODELS_DIR = PROJECT_ROOT / "models"
if "DATA_DIR" not in globals():
    DATA_DIR = PROJECT_ROOT / "data"


def _list_files(directory: Path, extension: str):
    if directory.exists():
        return sorted([f.name for f in directory.glob(f"*{extension}")])
    return []


model_files = _list_files(MODELS_DIR, ".pt")
data_files = _list_files(DATA_DIR, ".pkl")

exp_model_widget = widgets.Dropdown(
    options=model_files if model_files else ["Keine Modelle gefunden"],
    description="Modell:",
    style={"description_width": "initial"},
)

exp_data_widget = widgets.Dropdown(
    options=data_files if data_files else ["Keine Daten gefunden"],
    description="Daten:",
    style={"description_width": "initial"},
)

split_file_widget = widgets.Text(
    value="models/graph_split.pkl",
    description="Split Datei:",
    style={"description_width": "initial"},
)

graph_scope_widget = widgets.Dropdown(
    options=[("Testsplit", "test_split"), ("Gesamtdatensatz", "full_dataset")],
    value="test_split",
    description="Graph Scope:",
    style={"description_width": "initial"},
)

first_graph_per_component_widget = widgets.Checkbox(
    value=True,
    description="Erster Graph pro Compound",
    indent=False,
)

component_key_widget = widgets.Text(
    value="compound",
    description="Component Key:",
    style={"description_width": "initial"},
)

node_types_widget = widgets.SelectMultiple(
    options=["H", "C", "Others"],
    value=("H", "C"),
    description="Node Types:",
    style={"description_width": "initial"},
)

sparsity_widget = widgets.FloatSlider(
    value=0.90,
    min=0.50,
    max=0.99,
    step=0.01,
    description="Sparsity:",
    style={"description_width": "initial"},
    readout_format=".2f",
)

mask_baseline_widget = widgets.Dropdown(
    options=["match_ig_baseline", "zero", "mean"],
    value="match_ig_baseline",
    description="Mask Baseline:",
    style={"description_width": "initial"},
)

ig_baseline_widget = widgets.Dropdown(
    options=["scientific", "zero", "mean", "random", "min", "max"],
    value="scientific",
    description="IG Baseline:",
    style={"description_width": "initial"},
)

gnn_epochs_widget = widgets.IntSlider(
    value=200,
    min=50,
    max=500,
    step=50,
    description="GNN Epochs:",
    style={"description_width": "initial"},
)

gnn_lr_widget = widgets.FloatLogSlider(
    value=0.01,
    base=10,
    min=-4,
    max=-1,
    step=0.1,
    description="GNN LR:",
    style={"description_width": "initial"},
    readout_format=".4f",
)

gnn_explanation_type_widget = widgets.Dropdown(
    options=["phenomenon", "model"],
    value="phenomenon",
    description="GNN Type:",
    style={"description_width": "initial"},
)

ig_n_steps_widget = widgets.IntSlider(
    value=64,
    min=8,
    max=256,
    step=8,
    description="IG Steps:",
    style={"description_width": "initial"},
)

max_graphs_widget = widgets.IntText(
    value=0,
    description="Max Graphen (0=all):",
    style={"description_width": "initial"},
)

max_nodes_per_graph_widget = widgets.IntText(
    value=0,
    description="Max Nodes/Graph (0=all):",
    style={"description_width": "initial"},
)

include_edge_report_widget = widgets.Checkbox(
    value=True,
    description="GNN Edge Report",
    indent=False,
)

seed_widget = widgets.IntText(
    value=0,
    description="Seed:",
    style={"description_width": "initial"},
)

output_dir_widget = widgets.Text(
    value="results/experiments",
    description="Output Dir:",
    style={"description_width": "initial"},
)

controls = widgets.VBox(
    [
        widgets.HBox([exp_model_widget, exp_data_widget]),
        widgets.HBox([split_file_widget, graph_scope_widget]),
        widgets.HBox([first_graph_per_component_widget, component_key_widget]),
        widgets.HBox([node_types_widget, sparsity_widget]),
        widgets.HBox([mask_baseline_widget, ig_baseline_widget]),
        widgets.HBox([gnn_epochs_widget, gnn_lr_widget, gnn_explanation_type_widget]),
        widgets.HBox([ig_n_steps_widget, max_graphs_widget, max_nodes_per_graph_widget]),
        widgets.HBox([include_edge_report_widget, seed_widget, output_dir_widget]),
    ]
)

display(controls)


## Fidelity and Sparsity Metrics (Node-Level Regression)

Fuer jeden Knoten und jede Methode wird bei Ziel-Sparsity `s=0.90` zunaechst `k = max(1, ceil((1-s) * d))` (mit `d` = Feature-Anzahl) bestimmt.

- `S`: Top-k Features nach `|importance|`
- `actual_sparsity_feat = 1 - k/d`

Feature-Perturbationen am ausgewaehlten Knoten:

- `x_drop`: Features in `S` werden durch Baseline ersetzt
- `x_keep`: nur Features in `S` bleiben, Rest wird durch Baseline ersetzt

Mit `y_orig`, `y_drop`, `y_keep` und optional `y_true`:

- `fid_plus_model = |y_orig - y_drop|`
- `fid_minus_model = |y_orig - y_keep|`
- `fid_plus_error_delta = |y_drop - y_true| - |y_orig - y_true|`
- `fid_minus_error_delta = |y_keep - y_true| - |y_orig - y_true|`

Fuer den optionalen GNN-Edge-Report wird die Edge-Selektion global auf dem gesamten erklaerten Graphen durchgefuehrt:

- `E`: Anzahl aller beruecksichtigten Edges im Graphen
- `k_edge = max(1, ceil((1-s) * E))`, Ranking nach `|edge_mask|`
- Edge-`fid+`: Top-k Edges werden entfernt (`drop_selected`)
- Edge-`fid-`: Es bleiben nur die Top-k Edges erhalten (`keep_selected`)
- `actual_sparsity_edge = 1 - k_edge/E`

Die Aggregation erfolgt getrennt fuer `H` und `C`. Der faire Methodenvergleich nutzt nur die gemeinsame Node-Menge von GNNExplainer und IG.



In [4]:
from scripts.explainer.experiments_evaluation import (
    exp_load_eval_graph_indices,
    exp_build_mask_baseline,
    exp_topk_indices_from_importance,
    exp_predict_single_node,
    exp_feature_fidelity_for_node,
    exp_gnn_edge_fidelity_for_node,
    exp_extract_gnn_feature_importance,
    exp_extract_ig_feature_importance,
    select_scope_graph_indices,
    first_graph_indices_per_component,
    build_default_context,
    run_experiments_evaluation as _run_experiments_evaluation_core,
)


def run_experiments_evaluation(
    model_file=None,
    data_file=None,
    split_file=None,
    node_types=('H', 'C'),
    sparsity=0.90,
    mask_baseline_mode='match_ig_baseline',
    ig_baseline_mode='scientific',
    gnn_epochs=200,
    gnn_lr=0.01,
    gnn_explanation_type='phenomenon',
    ig_n_steps=64,
    graph_scope='test_split',
    first_graph_per_component=True,
    component_key='compound',
    max_graphs=None,
    max_nodes_per_graph=0,
    include_gnn_edge_report=True,
    output_dir='results/experiments',
    seed=0,
    verbose=True,
):
    model_name = model_file if model_file is not None else exp_model_widget.value
    data_name = data_file if data_file is not None else exp_data_widget.value

    if str(model_name).startswith('Keine') or str(data_name).startswith('Keine'):
        raise ValueError('Bitte g?ltige Modell- und Datendatei ausw?hlen.')

    split_eff = split_file if split_file is not None else split_file_widget.value

    context = build_default_context(
        project_root=PROJECT_ROOT,
        model_file=str(model_name),
        data_file=str(data_name),
        split_file=str(split_eff),
        output_dir=output_dir,
    )

    return _run_experiments_evaluation_core(
        context=context,
        node_types=node_types,
        sparsity=float(sparsity),
        mask_baseline_mode=str(mask_baseline_mode),
        ig_baseline_mode=str(ig_baseline_mode),
        gnn_epochs=int(gnn_epochs),
        gnn_lr=float(gnn_lr),
        gnn_explanation_type=str(gnn_explanation_type),
        gnn_use_custom_coeffs=False,
        gnn_coeffs=None,
        ig_n_steps=int(ig_n_steps),
        graph_scope=str(graph_scope),
        first_graph_per_component=bool(first_graph_per_component),
        component_key=str(component_key),
        max_graphs=max_graphs,
        max_nodes_per_graph=int(max_nodes_per_graph),
        include_gnn_edge_report=bool(include_gnn_edge_report),
        seed=int(seed),
        verbose=bool(verbose),
    )


In [5]:
run_button = widgets.Button(
    description='Experiments ausf?hren',
    button_style='success',
    icon='play'
)
run_output = widgets.Output()


def _run_clicked(_):
    with run_output:
        clear_output()
        try:
            global exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df
            max_graphs = int(max_graphs_widget.value)
            max_graphs = None if max_graphs <= 0 else max_graphs

            exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df = run_experiments_evaluation(
                node_types=tuple(node_types_widget.value),
                sparsity=float(sparsity_widget.value),
                mask_baseline_mode=mask_baseline_widget.value,
                ig_baseline_mode=ig_baseline_widget.value,
                gnn_epochs=int(gnn_epochs_widget.value),
                gnn_lr=float(gnn_lr_widget.value),
                gnn_explanation_type=gnn_explanation_type_widget.value,
                ig_n_steps=int(ig_n_steps_widget.value),
                graph_scope=graph_scope_widget.value,
                first_graph_per_component=bool(first_graph_per_component_widget.value),
                component_key=component_key_widget.value.strip() or 'compound',
                max_graphs=max_graphs,
                max_nodes_per_graph=int(max_nodes_per_graph_widget.value),
                include_gnn_edge_report=bool(include_edge_report_widget.value),
                output_dir=output_dir_widget.value,
                seed=int(seed_widget.value),
                verbose=True,
            )

            print('Summary by method/node type:')
            display(exp_summary_method_type_df)
            print('Fair comparison (shared nodes):')
            display(exp_summary_fair_df)
        except Exception as exc:
            print(f'? Fehler: {exc}')
            raise


run_button.on_click(_run_clicked)
display(run_button)
display(run_output)


Button(button_style='success', description='Experiments ausf?hren', icon='play', style=ButtonStyle())

Output()

In [6]:
# Optionaler Smoke-Run (bewusst auskommentiert):
# exp_node_df, exp_summary_method_type_df, exp_summary_fair_df, exp_gnn_edge_df = run_experiments_evaluation(
#     node_types=('H', 'C'),
#     sparsity=0.90,
#     mask_baseline_mode='match_ig_baseline',
#     ig_baseline_mode='scientific',
#     graph_scope='test_split',
#     first_graph_per_component=True,
#     component_key='compound',
#     max_graphs=2,
#     max_nodes_per_graph=5,
#     include_gnn_edge_report=True,
#     seed=0,
#     verbose=True,
# )
# display(exp_summary_method_type_df)
# display(exp_summary_fair_df)


### Hyperparameter Gridsearch für GNNExplainer

Diese Subsection fuehrt eine **staged Hyperparameter-Suche** fuer die vier GNNExplainer-Regularisierungen durch:

- **Stage A (2D):** `edge_size` x `edge_ent` bei neutraler Feature-Regularisierung
- **Stage B (2D):** `node_feat_size` x `node_feat_ent` bei neutraler Edge-Regularisierung
- **Stage C:** Kombination der besten `K` Kandidaten aus A und B (`K^2` Kombinationen)
- **Stage D (optional):** lokale Verfeinerung um das aktuell beste Set (eine log-Stufe nach unten/oben)

Die Suche verwendet **log-spaces** fuer die vier Koeffizienten, weil deren Einfluss typischerweise multiplikativ ist und auf mehreren Groessenordnungen stattfindet. Das ergibt stabilere und effizientere Raster als lineare Skalen.

Budget-Trennung:

- Suche auf `SEARCH_NODES` (schneller, konfigurierbar)
- Finale Bewertung auf `FINAL_NODES=150` Zielinstanzen


In [7]:
# Gridsearch Setup / Config
import time
import random
import itertools
import math
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from IPython.display import display
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer
from torch_geometric.explain.config import (
    ModelConfig,
    ModelMode,
    ModelReturnType,
    ModelTaskLevel,
)

from scripts.explainer.experiments_evaluation import (
    exp_extract_gnn_feature_importance,
    exp_feature_fidelity_for_node,
    exp_gnn_edge_fidelity_for_node,
    first_graph_indices_per_component,
    get_train_graph_indices,
    select_scope_graph_indices,
    build_default_context,
)
from scripts.explainer.explainer_utils import (
    NodeTypeRegressionWrapper,
    build_dataset,
    get_device,
    heterodata_to_dicts,
    load_config,
    load_stats,
    load_trained_model,
)


def _safe_widget_value(widget_name, fallback):
    widget = globals().get(widget_name)
    if widget is None:
        return fallback
    try:
        value = widget.value
    except Exception:
        return fallback
    if isinstance(value, str) and value.startswith("Keine"):
        return fallback
    return value


# File defaults (widgets if available)
MODEL_FILE = str(_safe_widget_value("exp_model_widget", "SAGEConv_best_model.pt"))
DATA_FILE = str(_safe_widget_value("exp_data_widget", "all_graphs.pkl"))
SPLIT_FILE = str(_safe_widget_value("split_file_widget", "models/graph_split.pkl"))

# Main search budget
SEARCH_NODES = 30
FINAL_NODES = 150
SEEDS = [0, 1, 2]

# Explainer optimization
EPOCHS_SEARCH = 150
EPOCHS_FINAL = 250
LR = float(_safe_widget_value("gnn_lr_widget", 0.01))

# Stage control
TOP_K_STAGE = 3
STAGE_D_BUDGET = 0
STAGE_D_USE_SEARCH_EPOCHS = True

# Progress tracking
PROGRESS_ENABLED = True
PROGRESS_USE_TQDM = True
PROGRESS_GRANULARITY = "stage_config_seed"

# Evaluation scope
NODE_TYPES = tuple(_safe_widget_value("node_types_widget", ("H", "C")))
GRAPH_SCOPE = str(_safe_widget_value("graph_scope_widget", "test_split"))
FIRST_GRAPH_PER_COMPONENT = bool(_safe_widget_value("first_graph_per_component_widget", True))
COMPONENT_KEY = str(_safe_widget_value("component_key_widget", "compound"))

# Fidelity config
SPARSITY = float(_safe_widget_value("sparsity_widget", 0.90))
MASK_BASELINE_MODE = str(_safe_widget_value("mask_baseline_widget", "match_ig_baseline"))
IG_BASELINE_MODE = str(_safe_widget_value("ig_baseline_widget", "scientific"))
GNN_EXPLANATION_TYPE = str(_safe_widget_value("gnn_explanation_type_widget", "phenomenon"))

# Selection reporting config (fixed but editable)
THRESHOLD_CONFIG = {
    "feature_threshold": 0.5,
    "edge_threshold": 0.5,
    "min_keep": 1,
    "use_abs": True,
}
TOPK_CONFIG = {
    "enabled": False,
    "topk_features": None,
    "topk_edges": None,
}

# Log-space grids
GRID_EDGE_SIZE = np.logspace(-6, -1, 6)
GRID_EDGE_ENT = np.logspace(-6, 0, 7)
GRID_NODE_FEAT_SIZE = np.logspace(-6, 0, 7)
GRID_NODE_FEAT_ENT = np.logspace(-6, -1, 6)

# Neutral regularization settings for staged search
NEUTRAL_EDGE_SIZE = 1e-12
NEUTRAL_EDGE_ENT = 1e-12
NEUTRAL_NODE_FEAT_SIZE = 1e-12
NEUTRAL_NODE_FEAT_ENT = 1e-12

print("Gridsearch config loaded.")
print(f"MODEL_FILE={MODEL_FILE}")
print(f"DATA_FILE={DATA_FILE}")
print(f"SPLIT_FILE={SPLIT_FILE}")
print(f"SEARCH_NODES={SEARCH_NODES}, FINAL_NODES={FINAL_NODES}, SEEDS={SEEDS}")



Gridsearch config loaded.
MODEL_FILE=GraphConv_best_model.pt
DATA_FILE=all_graphs_with_length.pkl
SPLIT_FILE=models/graph_split.pkl
SEARCH_NODES=30, FINAL_NODES=150, SEEDS=[0, 1, 2]


In [8]:
# Gridsearch helper functions

def set_all_seeds(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def build_gnnexplainer_algorithm(epochs, lr, coeffs):
    coeffs = {key: float(value) for key, value in dict(coeffs).items()}

    def _coeffs_applied(algo):
        algo_coeffs = getattr(algo, "coeffs", None)
        if not isinstance(algo_coeffs, dict):
            return False
        for key, expected in coeffs.items():
            if key not in algo_coeffs:
                return False
            if not np.isclose(float(algo_coeffs[key]), float(expected), rtol=0.0, atol=0.0):
                return False
        return True

    # Preferred on modern PyG: coefficients as explicit kwargs.
    try:
        algo = GNNExplainer(epochs=int(epochs), lr=float(lr), **coeffs)
        if _coeffs_applied(algo):
            return algo
    except TypeError:
        pass

    # Robust fallback: instantiate defaults and patch coeff dict in-place.
    algo = GNNExplainer(epochs=int(epochs), lr=float(lr))
    algo_coeffs = getattr(algo, "coeffs", None)
    if isinstance(algo_coeffs, dict):
        algo_coeffs.update(coeffs)
    if _coeffs_applied(algo):
        return algo

    raise TypeError("Could not apply custom coeffs to this GNNExplainer version.")


def safe_tqdm(iterable, total, desc, enabled=True, use_tqdm=True):
    if not bool(enabled):
        return iterable
    if bool(use_tqdm):
        try:
            from tqdm.auto import tqdm

            return tqdm(iterable, total=total, desc=str(desc), leave=False)
        except Exception:
            pass
    return iterable


def _clone_edge_attr_dict(edge_attr_dict):
    if edge_attr_dict is None:
        return None
    return {
        edge_type: (attrs.clone() if attrs is not None else None)
        for edge_type, attrs in edge_attr_dict.items()
    }


def _coeff_signature(coeffs):
    return (
        float(coeffs["edge_size"]),
        float(coeffs["edge_ent"]),
        float(coeffs["node_feat_size"]),
        float(coeffs["node_feat_ent"]),
    )


def load_eval_artifacts(
    project_root,
    model_file,
    data_file,
    split_file,
    node_types,
    graph_scope,
    first_graph_per_component,
    component_key,
):
    context = build_default_context(
        project_root=Path(project_root),
        model_file=str(model_file),
        data_file=str(data_file),
        split_file=str(split_file),
        output_dir="results/experiments",
    )

    device = get_device(None)
    config = load_config(str(context.config_path))
    norm_stats, edge_stats = load_stats(str(context.norm_stats_path), str(context.edge_stats_path))
    dataset = build_dataset(str(context.data_path), config, norm_stats=norm_stats, edge_stats=edge_stats)
    base_model = load_trained_model(str(context.model_path), config, device)

    graph_indices = select_scope_graph_indices(dataset=dataset, split_path=context.split_path, graph_scope=graph_scope)
    if first_graph_per_component:
        graph_indices = first_graph_indices_per_component(
            dataset=dataset,
            graph_indices=graph_indices,
            component_key=component_key,
        )

    train_graph_indices = get_train_graph_indices(context.split_path, total_graphs=len(dataset))
    clean_node_types = tuple(nt for nt in node_types if nt in ("H", "C", "Others"))

    return {
        "context": context,
        "device": device,
        "config": config,
        "dataset": dataset,
        "base_model": base_model,
        "eval_graph_indices": [int(i) for i in graph_indices],
        "train_graph_indices": train_graph_indices,
        "node_types": clean_node_types,
    }


def collect_candidate_nodes(dataset, eval_graph_indices, node_types, explanation_type="phenomenon"):
    rows = []
    explanation_type = str(explanation_type)

    for graph_idx in sorted(int(i) for i in eval_graph_indices):
        data = dataset[graph_idx]

        for node_type in node_types:
            if node_type not in data.node_types:
                continue

            x = data[node_type].x
            num_nodes = int(x.size(0))
            if num_nodes <= 0:
                continue

            y_tensor = getattr(data[node_type], "y", None)
            y_flat = y_tensor.reshape(-1) if y_tensor is not None else None

            if explanation_type == "phenomenon" and y_flat is None:
                continue

            for node_idx in range(num_nodes):
                target_value = float("nan")
                if y_flat is not None and node_idx < int(y_flat.numel()):
                    value = y_flat[node_idx]
                    target_value = float(value.item())

                if explanation_type == "phenomenon" and not np.isfinite(target_value):
                    continue

                rows.append(
                    {
                        "graph_idx": int(graph_idx),
                        "node_type": str(node_type),
                        "node_idx": int(node_idx),
                        "target_value": float(target_value),
                    }
                )

    return rows


def sample_target_nodes(candidates, search_nodes, final_nodes, seed):
    if not candidates:
        raise ValueError("No candidate nodes available for search/evaluation.")

    rng = random.Random(int(seed))
    indices = list(range(len(candidates)))
    rng.shuffle(indices)
    shuffled = [candidates[i] for i in indices]

    if int(final_nodes) <= 0:
        final_count = len(shuffled)
    else:
        final_count = min(int(final_nodes), len(shuffled))

    final_selected = shuffled[:final_count]

    if int(search_nodes) <= 0:
        search_count = final_count
    else:
        search_count = min(int(search_nodes), final_count)

    search_selected = final_selected[:search_count]
    return search_selected, final_selected


def build_base_prediction_cache(base_model, dataset, target_nodes, device):
    graph_to_nodes = {}
    for row in target_nodes:
        graph_to_nodes.setdefault(int(row["graph_idx"]), []).append(row)

    graph_cache = {}
    pred_cache = {}

    for graph_idx in sorted(graph_to_nodes.keys()):
        data = dataset[graph_idx].to(device)
        x_dict_raw, edge_index_dict, edge_attr_dict_raw, y_dict = heterodata_to_dicts(data)

        x_dict = {nt: feat.clone() for nt, feat in x_dict_raw.items()}
        edge_attr_dict = _clone_edge_attr_dict(edge_attr_dict_raw)

        # The model forward mutates x_dict/edge_attr_dict in-place; use throwaway copies for pred cache.
        with torch.no_grad():
            pred_dict = base_model(
                {nt: feat.clone() for nt, feat in x_dict.items()},
                edge_index_dict,
                _clone_edge_attr_dict(edge_attr_dict),
            )

        graph_cache[graph_idx] = {
            "x_dict": x_dict,
            "edge_index_dict": edge_index_dict,
            "edge_attr_dict": edge_attr_dict,
            "y_dict": y_dict,
            "pred_dict": pred_dict,
        }

        for row in graph_to_nodes[graph_idx]:
            node_type = row["node_type"]
            node_idx = int(row["node_idx"])
            pred_tensor = pred_dict.get(node_type)
            if pred_tensor is None:
                continue
            flat = pred_tensor.reshape(-1)
            if 0 <= node_idx < int(flat.size(0)):
                pred_cache[(graph_idx, node_type, node_idx)] = float(flat[node_idx].item())

    return graph_cache, pred_cache


def count_selected_features(mask_values, threshold_cfg, topk_cfg):
    values = np.asarray(mask_values, dtype=float).reshape(-1)
    if values.size == 0:
        return 0

    use_abs = bool(threshold_cfg.get("use_abs", True))
    min_keep = max(0, int(threshold_cfg.get("min_keep", 0)))
    scores = np.abs(values) if use_abs else values

    if bool(topk_cfg.get("enabled", False)) and topk_cfg.get("topk_features") is not None:
        k = max(0, int(topk_cfg["topk_features"]))
        selected = min(k, int(scores.size))
        if min_keep > 0:
            selected = max(min_keep, selected)
        return int(min(selected, int(scores.size)))

    thr = float(threshold_cfg.get("feature_threshold", 0.5))
    selected = int(np.sum(np.isfinite(scores) & (scores >= thr)))
    if min_keep > 0:
        selected = max(min_keep, selected)
    return int(min(selected, int(scores.size)))


def count_selected_edges(edge_mask_dict, edge_index_dict, node_type, node_idx, threshold_cfg, topk_cfg):
    # Compatibility with existing call sites; edge counting is global across the graph.
    del node_type, node_idx

    use_abs = bool(threshold_cfg.get("use_abs", True))
    min_keep = max(0, int(threshold_cfg.get("min_keep", 0)))

    edge_scores = []
    for edge_type, edge_mask in edge_mask_dict.items():
        if edge_mask is None or edge_type not in edge_index_dict:
            continue

        edge_index = edge_index_dict[edge_type]
        mask_vals = edge_mask.view(-1).detach().cpu().numpy().reshape(-1)

        limit = min(mask_vals.shape[0], int(edge_index.size(1)))
        for pos in range(limit):
            score = float(mask_vals[pos])
            edge_scores.append(abs(score) if use_abs else score)

    if not edge_scores:
        return 0

    scores = np.asarray(edge_scores, dtype=float)

    if bool(topk_cfg.get("enabled", False)) and topk_cfg.get("topk_edges") is not None:
        k = max(0, int(topk_cfg["topk_edges"]))
        selected = min(k, int(scores.size))
        if min_keep > 0:
            selected = max(min_keep, selected)
        return int(min(selected, int(scores.size)))

    thr = float(threshold_cfg.get("edge_threshold", 0.5))
    selected = int(np.sum(np.isfinite(scores) & (scores >= thr)))
    if min_keep > 0:
        selected = max(min_keep, selected)
    return int(min(selected, int(scores.size)))


def _median(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    return float(np.median(arr)) if arr.size > 0 else float("nan")


def _iqr(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return float("nan")
    q75, q25 = np.percentile(arr, [75, 25])
    return float(q75 - q25)


def evaluate_single_config(
    stage,
    coeffs,
    epochs,
    lr,
    seed,
    target_nodes,
    base_model,
    graph_cache,
    pred_cache,
    dataset,
    train_graph_indices,
    node_types,
    sparsity,
    mask_baseline_mode,
    ig_baseline_mode,
    explanation_type,
    threshold_cfg,
    topk_cfg,
    median_cache,
    elem_distribution_cache,
    baseline_cache_dir,
):
    set_all_seeds(seed)
    t0 = time.perf_counter()

    explainers = {}
    model_config = ModelConfig(
        mode=ModelMode.regression,
        task_level=ModelTaskLevel.node,
        return_type=ModelReturnType.raw,
    )
    for node_type in node_types:
        wrapped_model = NodeTypeRegressionWrapper(base_model, node_type)
        algorithm = build_gnnexplainer_algorithm(epochs=epochs, lr=lr, coeffs=coeffs)
        explainers[node_type] = Explainer(
            model=wrapped_model,
            algorithm=algorithm,
            explanation_type=str(explanation_type),
            model_config=model_config,
            node_mask_type="attributes",
            edge_mask_type="object",
        )

    combined_vals = []
    abs_delta_vals = []
    edges_selected_vals = []
    feats_selected_vals = []
    failures = 0
    error_counts = {}
    fallback_model_target_count = 0
    error_first_message = ""

    for row in target_nodes:
        graph_idx = int(row["graph_idx"])
        node_type = str(row["node_type"])
        node_idx = int(row["node_idx"])
        target_value = float(row.get("target_value", float("nan")))

        if node_type not in explainers:
            continue
        if graph_idx not in graph_cache:
            failures += 1
            key = "graph_cache_missing"
            error_counts[key] = int(error_counts.get(key, 0)) + 1
            continue

        cache_item = graph_cache[graph_idx]
        x_dict_base = cache_item["x_dict"]
        edge_index_dict = cache_item["edge_index_dict"]
        edge_attr_base = cache_item["edge_attr_dict"]
        y_dict = cache_item["y_dict"]

        pred_orig = pred_cache.get((graph_idx, node_type, node_idx))
        if pred_orig is None:
            failures += 1
            key = "pred_orig_missing"
            error_counts[key] = int(error_counts.get(key, 0)) + 1
            continue

        gnn_target = None
        if str(explanation_type) == "phenomenon":
            y_tensor = y_dict.get(node_type)
            if y_tensor is None:
                failures += 1
                key = "target_missing"
                error_counts[key] = int(error_counts.get(key, 0)) + 1
                continue
            y_tensor = y_tensor.reshape(-1)
            if node_idx >= int(y_tensor.size(0)) or torch.isnan(y_tensor[node_idx]):
                failures += 1
                key = "target_nan_or_oob"
                error_counts[key] = int(error_counts.get(key, 0)) + 1
                continue
            gnn_target = y_tensor

        try:
            try:
                explanation = explainers[node_type](
                    {nt: feat.clone() for nt, feat in x_dict_base.items()},
                    edge_index_dict,
                    edge_attr_dict=_clone_edge_attr_dict(edge_attr_base),
                    target=gnn_target,
                    index=node_idx,
                )
            except Exception as exc_explain:
                # Compatibility fallback: indexed node-level calls can require scalar targets.
                if str(explanation_type) == "phenomenon" and gnn_target is not None:
                    try:
                        explanation = explainers[node_type](
                            {nt: feat.clone() for nt, feat in x_dict_base.items()},
                            edge_index_dict,
                            edge_attr_dict=_clone_edge_attr_dict(edge_attr_base),
                            target=gnn_target[node_idx],
                            index=node_idx,
                        )
                        fallback_model_target_count += 1
                    except Exception:
                        raise exc_explain
                else:
                    raise exc_explain

            feature_importance = exp_extract_gnn_feature_importance(
                explanation,
                node_type=node_type,
                node_idx=node_idx,
            )

            feature_metrics = exp_feature_fidelity_for_node(
                base_model=base_model,
                x_dict=x_dict_base,
                edge_index_dict=edge_index_dict,
                edge_attr_dict=edge_attr_base,
                node_type=node_type,
                node_idx=node_idx,
                importance=feature_importance,
                sparsity=float(sparsity),
                mask_baseline_mode=str(mask_baseline_mode),
                ig_baseline_mode=str(ig_baseline_mode),
                dataset=dataset,
                train_graph_indices=train_graph_indices,
                median_cache=median_cache,
                elem_distribution_cache=elem_distribution_cache,
                cache_dir=baseline_cache_dir,
                pred_orig=float(pred_orig),
                target_value=float(target_value),
            )

            edge_metrics = exp_gnn_edge_fidelity_for_node(
                base_model=base_model,
                x_dict=x_dict_base,
                edge_index_dict=edge_index_dict,
                edge_attr_dict=edge_attr_base,
                edge_mask_dict=explanation.edge_mask_dict,
                node_type=node_type,
                node_idx=node_idx,
                sparsity=float(sparsity),
                pred_orig=float(pred_orig),
                target_value=float(target_value),
            )

            feat_fid = float(feature_metrics.get("fid_plus_model", float("nan")))
            edge_fid = float(edge_metrics.get("fid_plus_model", float("nan"))) if edge_metrics is not None else float("nan")
            combined_fidelity = float(np.nanmean([feat_fid, edge_fid]))

            feat_delta = abs(float(feature_metrics.get("fid_plus_error_delta", float("nan"))))
            edge_delta = (
                abs(float(edge_metrics.get("fid_plus_error_delta", float("nan"))))
                if edge_metrics is not None
                else float("nan")
            )
            abs_delta = float(np.nanmean([feat_delta, edge_delta]))

            if not np.isfinite(combined_fidelity):
                failures += 1
                key = "nonfinite_combined_fidelity"
                error_counts[key] = int(error_counts.get(key, 0)) + 1
                continue

            feats_selected = count_selected_features(feature_importance, threshold_cfg=threshold_cfg, topk_cfg=topk_cfg)
            edges_selected = count_selected_edges(
                explanation.edge_mask_dict,
                edge_index_dict,
                node_type=node_type,
                node_idx=node_idx,
                threshold_cfg=threshold_cfg,
                topk_cfg=topk_cfg,
            )

            combined_vals.append(combined_fidelity)
            abs_delta_vals.append(abs_delta)
            feats_selected_vals.append(float(feats_selected))
            edges_selected_vals.append(float(edges_selected))
        except Exception as exc:
            failures += 1
            key = f"explain_failed:{type(exc).__name__}"
            error_counts[key] = int(error_counts.get(key, 0)) + 1
            if not error_first_message:
                error_first_message = f"{type(exc).__name__}: {exc}"

    runtime_sec = float(time.perf_counter() - t0)

    if error_counts:
        error_top_reason, error_top_count = sorted(error_counts.items(), key=lambda kv: (-int(kv[1]), str(kv[0])))[0]
    else:
        error_top_reason, error_top_count = "", 0

    row = {
        "stage": str(stage),
        "edge_size": float(coeffs["edge_size"]),
        "edge_ent": float(coeffs["edge_ent"]),
        "node_feat_size": float(coeffs["node_feat_size"]),
        "node_feat_ent": float(coeffs["node_feat_ent"]),
        "lr": float(lr),
        "epochs": int(epochs),
        "seed": int(seed),
        "fidelity_median": _median(combined_vals),
        "fidelity_iqr": _iqr(combined_vals),
        "abs_delta_median": _median(abs_delta_vals),
        "edges_selected_median": _median(edges_selected_vals),
        "feats_selected_median": _median(feats_selected_vals),
        "runtime_sec": runtime_sec,
        "n_nodes_evaluated": int(len(combined_vals)),
        "n_nodes_requested": int(len(target_nodes)),
        "n_failures": int(failures),
        "error_top_reason": str(error_top_reason),
        "error_top_count": int(error_top_count),
        "error_first_message": str(error_first_message),
        "fallback_model_target_count": int(fallback_model_target_count),
    }
    return row


def run_stage_grid(
    stage_name,
    coeff_dicts,
    epochs,
    seeds,
    target_nodes,
    base_model,
    graph_cache,
    pred_cache,
    dataset,
    train_graph_indices,
    node_types,
    sparsity,
    mask_baseline_mode,
    ig_baseline_mode,
    explanation_type,
    threshold_cfg,
    topk_cfg,
    median_cache,
    elem_distribution_cache,
    baseline_cache_dir,
    lr,
):
    rows = []

    progress_enabled = bool(globals().get("PROGRESS_ENABLED", True))
    progress_use_tqdm = bool(globals().get("PROGRESS_USE_TQDM", True))
    progress_granularity = str(globals().get("PROGRESS_GRANULARITY", "stage_config_seed"))
    show_config_seed_progress = progress_enabled and progress_granularity == "stage_config_seed"

    total_runs = int(len(coeff_dicts) * len(seeds))
    t_stage = time.perf_counter()
    if progress_enabled:
        print(f"[{stage_name}] start: {total_runs} runs")

    run_iter = itertools.product(coeff_dicts, seeds)
    run_iter = safe_tqdm(
        run_iter,
        total=total_runs,
        desc=f"Stage {stage_name}",
        enabled=show_config_seed_progress,
        use_tqdm=progress_use_tqdm,
    )

    for coeffs, seed in run_iter:
        rows.append(
            evaluate_single_config(
                stage=stage_name,
                coeffs=coeffs,
                epochs=epochs,
                lr=lr,
                seed=seed,
                target_nodes=target_nodes,
                base_model=base_model,
                graph_cache=graph_cache,
                pred_cache=pred_cache,
                dataset=dataset,
                train_graph_indices=train_graph_indices,
                node_types=node_types,
                sparsity=sparsity,
                mask_baseline_mode=mask_baseline_mode,
                ig_baseline_mode=ig_baseline_mode,
                explanation_type=explanation_type,
                threshold_cfg=threshold_cfg,
                topk_cfg=topk_cfg,
                median_cache=median_cache,
                elem_distribution_cache=elem_distribution_cache,
                baseline_cache_dir=baseline_cache_dir,
            )
        )

    stage_runtime = float(time.perf_counter() - t_stage)
    if progress_enabled:
        print(f"[{stage_name}] done: {len(rows)}/{total_runs} runs in {stage_runtime:.1f}s")

    return pd.DataFrame(rows)


def aggregate_stage_scores(df_stage):
    if df_stage is None or df_stage.empty:
        return pd.DataFrame()

    group_cols = [
        "stage",
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "lr",
        "epochs",
    ]
    metric_cols = [
        "fidelity_median",
        "fidelity_iqr",
        "abs_delta_median",
        "edges_selected_median",
        "feats_selected_median",
        "runtime_sec",
        "n_nodes_evaluated",
        "n_nodes_requested",
        "n_failures",
    ]

    rows = []
    for keys, group_df in df_stage.groupby(group_cols, dropna=False):
        row = {col: val for col, val in zip(group_cols, keys)}
        for metric in metric_cols:
            values = pd.to_numeric(group_df[metric], errors="coerce").dropna()
            row[metric] = float(values.median()) if not values.empty else float("nan")

        fidelity_values = pd.to_numeric(group_df["fidelity_median"], errors="coerce").dropna()
        row["fidelity_std"] = float(fidelity_values.std(ddof=0)) if not fidelity_values.empty else float("nan")
        row["seed_runs"] = int(group_df["seed"].nunique())
        row["selection_sum"] = float(row["edges_selected_median"] + row["feats_selected_median"])
        rows.append(row)

    out = pd.DataFrame(rows)
    return out.sort_values(["stage", "fidelity_median"], ascending=[True, False]).reset_index(drop=True)


def select_top_k_configs(df_stage_agg, k=3):
    if df_stage_agg is None or df_stage_agg.empty:
        return pd.DataFrame()

    work = df_stage_agg.copy()
    if "selection_sum" not in work.columns:
        work["selection_sum"] = work["edges_selected_median"] + work["feats_selected_median"]

    work = work.sort_values(
        ["fidelity_median", "selection_sum", "runtime_sec"],
        ascending=[False, True, True],
    )
    return work.head(max(0, int(k))).reset_index(drop=True)


def build_stage_c_candidates(top_a_df, top_b_df):
    if top_a_df is None or top_b_df is None or top_a_df.empty or top_b_df.empty:
        return []

    candidates = []
    seen = set()
    for row_a in top_a_df.itertuples(index=False):
        for row_b in top_b_df.itertuples(index=False):
            coeffs = {
                "edge_size": float(row_a.edge_size),
                "edge_ent": float(row_a.edge_ent),
                "node_feat_size": float(row_b.node_feat_size),
                "node_feat_ent": float(row_b.node_feat_ent),
            }
            sig = _coeff_signature(coeffs)
            if sig in seen:
                continue
            seen.add(sig)
            candidates.append(coeffs)
    return candidates


def build_stage_d_candidates(best_cfg, grids, seen_cfgs, budget):
    budget = int(budget)
    if budget <= 0:
        return []

    def neighbor_values(grid_vals, current_val):
        grid_vals = np.asarray(grid_vals, dtype=float)
        cur = max(float(current_val), 1e-12)
        idx = int(np.argmin(np.abs(np.log10(grid_vals) - np.log10(cur))))
        idxs = [i for i in [idx - 1, idx, idx + 1] if 0 <= i < grid_vals.size]
        return sorted({float(grid_vals[i]) for i in idxs})

    edge_size_vals = neighbor_values(grids["edge_size"], best_cfg["edge_size"])
    edge_ent_vals = neighbor_values(grids["edge_ent"], best_cfg["edge_ent"])
    node_feat_size_vals = neighbor_values(grids["node_feat_size"], best_cfg["node_feat_size"])
    node_feat_ent_vals = neighbor_values(grids["node_feat_ent"], best_cfg["node_feat_ent"])

    candidates = []
    for es, ee, nfs, nfe in itertools.product(
        edge_size_vals,
        edge_ent_vals,
        node_feat_size_vals,
        node_feat_ent_vals,
    ):
        coeffs = {
            "edge_size": float(es),
            "edge_ent": float(ee),
            "node_feat_size": float(nfs),
            "node_feat_ent": float(nfe),
        }
        sig = _coeff_signature(coeffs)
        if sig in seen_cfgs:
            continue
        candidates.append(coeffs)

    candidates = sorted(
        candidates,
        key=lambda c: (c["edge_size"], c["edge_ent"], c["node_feat_size"], c["node_feat_ent"]),
    )
    return candidates[:budget]


def compute_pareto_front(df_agg):
    if df_agg is None or df_agg.empty:
        return pd.DataFrame()

    work = df_agg.copy()
    if "selection_sum" not in work.columns:
        work["selection_sum"] = work["edges_selected_median"] + work["feats_selected_median"]

    work = work[
        np.isfinite(pd.to_numeric(work["fidelity_median"], errors="coerce"))
        & np.isfinite(pd.to_numeric(work["selection_sum"], errors="coerce"))
    ].reset_index(drop=True)

    keep_indices = []
    for i, row_i in work.iterrows():
        dominated = False
        for j, row_j in work.iterrows():
            if i == j:
                continue

            better_or_equal = (
                float(row_j["fidelity_median"]) >= float(row_i["fidelity_median"])
                and float(row_j["selection_sum"]) <= float(row_i["selection_sum"])
            )
            strictly_better = (
                float(row_j["fidelity_median"]) > float(row_i["fidelity_median"])
                or float(row_j["selection_sum"]) < float(row_i["selection_sum"])
            )
            if better_or_equal and strictly_better:
                dominated = True
                break

        if not dominated:
            keep_indices.append(i)

    pareto = work.loc[keep_indices].copy()
    pareto = pareto.sort_values(["fidelity_median", "selection_sum"], ascending=[False, True]).reset_index(drop=True)
    return pareto







In [9]:
# Run staged gridsearch (A/B/C/(D)) + final evaluation
set_all_seeds(SEEDS[0])

artifacts = load_eval_artifacts(
    project_root=PROJECT_ROOT,
    model_file=MODEL_FILE,
    data_file=DATA_FILE,
    split_file=SPLIT_FILE,
    node_types=NODE_TYPES,
    graph_scope=GRAPH_SCOPE,
    first_graph_per_component=FIRST_GRAPH_PER_COMPONENT,
    component_key=COMPONENT_KEY,
)

dataset = artifacts["dataset"]
base_model = artifacts["base_model"]
device = artifacts["device"]
eval_graph_indices = artifacts["eval_graph_indices"]
train_graph_indices = artifacts["train_graph_indices"]
node_types_eff = artifacts["node_types"]
context = artifacts["context"]

candidate_nodes = collect_candidate_nodes(
    dataset=dataset,
    eval_graph_indices=eval_graph_indices,
    node_types=node_types_eff,
    explanation_type=GNN_EXPLANATION_TYPE,
)

if not candidate_nodes:
    raise RuntimeError("No valid candidate nodes found. Check scope/node types/targets.")

search_target_nodes, final_target_nodes = sample_target_nodes(
    candidates=candidate_nodes,
    search_nodes=SEARCH_NODES,
    final_nodes=FINAL_NODES,
    seed=SEEDS[0],
)

if len(final_target_nodes) < int(FINAL_NODES):
    print(
        f"Warning: only {len(final_target_nodes)} valid nodes available "
        f"(requested FINAL_NODES={FINAL_NODES})."
    )

print(f"Candidate nodes: {len(candidate_nodes)}")
print(f"Search nodes: {len(search_target_nodes)}")
print(f"Final nodes: {len(final_target_nodes)}")

# Cache graph data + base predictions for selected final nodes.
graph_cache, pred_cache = build_base_prediction_cache(
    base_model=base_model,
    dataset=dataset,
    target_nodes=final_target_nodes,
    device=device,
)

# Baseline caches for scientific baseline mode.
median_cache = {}
elem_distribution_cache = {}
if str(IG_BASELINE_MODE) == "scientific":
    try:
        from scripts.explainer.baselines import (
            load_or_compute_element_distribution,
            load_or_compute_medians,
        )

        baseline_cache_dir = context.output_dir.parent / "baselines"
        for nt in node_types_eff:
            try:
                median_cache[nt] = load_or_compute_medians(
                    dataset=dataset,
                    train_graph_indices=train_graph_indices,
                    node_type=nt,
                    cache_dir=str(baseline_cache_dir),
                )
            except Exception:
                pass
        try:
            elem_distribution_cache["value"] = load_or_compute_element_distribution(
                dataset=dataset,
                train_graph_indices=train_graph_indices,
                cache_dir=str(baseline_cache_dir),
            )
        except Exception:
            pass
    except Exception:
        baseline_cache_dir = context.output_dir.parent / "baselines"
else:
    baseline_cache_dir = context.output_dir.parent / "baselines"

# Stage A: edge regularization (node-feature regularization neutral)
stage_a_coeffs = [
    {
        "edge_size": float(edge_size),
        "edge_ent": float(edge_ent),
        "node_feat_size": float(NEUTRAL_NODE_FEAT_SIZE),
        "node_feat_ent": float(NEUTRAL_NODE_FEAT_ENT),
    }
    for edge_size, edge_ent in itertools.product(GRID_EDGE_SIZE, GRID_EDGE_ENT)
]

stage_a_df = run_stage_grid(
    stage_name="A",
    coeff_dicts=stage_a_coeffs,
    epochs=EPOCHS_SEARCH,
    seeds=SEEDS,
    target_nodes=search_target_nodes,
    base_model=base_model,
    graph_cache=graph_cache,
    pred_cache=pred_cache,
    dataset=dataset,
    train_graph_indices=train_graph_indices,
    node_types=node_types_eff,
    sparsity=SPARSITY,
    mask_baseline_mode=MASK_BASELINE_MODE,
    ig_baseline_mode=IG_BASELINE_MODE,
    explanation_type=GNN_EXPLANATION_TYPE,
    threshold_cfg=THRESHOLD_CONFIG,
    topk_cfg=TOPK_CONFIG,
    median_cache=median_cache,
    elem_distribution_cache=elem_distribution_cache,
    baseline_cache_dir=baseline_cache_dir,
    lr=LR,
)
stage_a_summary = aggregate_stage_scores(stage_a_df)
stage_a_top_df = select_top_k_configs(stage_a_summary, k=TOP_K_STAGE)

# Stage B: node-feature regularization (edge regularization neutral)
stage_b_coeffs = [
    {
        "edge_size": float(NEUTRAL_EDGE_SIZE),
        "edge_ent": float(NEUTRAL_EDGE_ENT),
        "node_feat_size": float(node_feat_size),
        "node_feat_ent": float(node_feat_ent),
    }
    for node_feat_size, node_feat_ent in itertools.product(GRID_NODE_FEAT_SIZE, GRID_NODE_FEAT_ENT)
]

stage_b_df = run_stage_grid(
    stage_name="B",
    coeff_dicts=stage_b_coeffs,
    epochs=EPOCHS_SEARCH,
    seeds=SEEDS,
    target_nodes=search_target_nodes,
    base_model=base_model,
    graph_cache=graph_cache,
    pred_cache=pred_cache,
    dataset=dataset,
    train_graph_indices=train_graph_indices,
    node_types=node_types_eff,
    sparsity=SPARSITY,
    mask_baseline_mode=MASK_BASELINE_MODE,
    ig_baseline_mode=IG_BASELINE_MODE,
    explanation_type=GNN_EXPLANATION_TYPE,
    threshold_cfg=THRESHOLD_CONFIG,
    topk_cfg=TOPK_CONFIG,
    median_cache=median_cache,
    elem_distribution_cache=elem_distribution_cache,
    baseline_cache_dir=baseline_cache_dir,
    lr=LR,
)
stage_b_summary = aggregate_stage_scores(stage_b_df)
stage_b_top_df = select_top_k_configs(stage_b_summary, k=TOP_K_STAGE)

# Stage C: combine top-K from A and B
stage_c_coeffs = build_stage_c_candidates(stage_a_top_df, stage_b_top_df)
if stage_c_coeffs:
    stage_c_df = run_stage_grid(
        stage_name="C",
        coeff_dicts=stage_c_coeffs,
        epochs=EPOCHS_SEARCH,
        seeds=SEEDS,
        target_nodes=search_target_nodes,
        base_model=base_model,
        graph_cache=graph_cache,
        pred_cache=pred_cache,
        dataset=dataset,
        train_graph_indices=train_graph_indices,
        node_types=node_types_eff,
        sparsity=SPARSITY,
        mask_baseline_mode=MASK_BASELINE_MODE,
        ig_baseline_mode=IG_BASELINE_MODE,
        explanation_type=GNN_EXPLANATION_TYPE,
        threshold_cfg=THRESHOLD_CONFIG,
        topk_cfg=TOPK_CONFIG,
        median_cache=median_cache,
        elem_distribution_cache=elem_distribution_cache,
        baseline_cache_dir=baseline_cache_dir,
        lr=LR,
    )
    stage_c_summary = aggregate_stage_scores(stage_c_df)
else:
    stage_c_df = pd.DataFrame()
    stage_c_summary = pd.DataFrame()

raw_parts = [df for df in [stage_a_df, stage_b_df, stage_c_df] if not df.empty]
summary_parts = [df for df in [stage_a_summary, stage_b_summary, stage_c_summary] if not df.empty]

search_summary_df = pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame()

# Stage D: optional local refinement around current best config
if STAGE_D_BUDGET > 0 and not search_summary_df.empty:
    best_pre_df = select_top_k_configs(search_summary_df, k=1)
    best_pre = best_pre_df.iloc[0].to_dict()

    seen_cfgs = {
        _coeff_signature(
            {
                "edge_size": row.edge_size,
                "edge_ent": row.edge_ent,
                "node_feat_size": row.node_feat_size,
                "node_feat_ent": row.node_feat_ent,
            }
        )
        for row in search_summary_df.itertuples(index=False)
    }

    stage_d_coeffs = build_stage_d_candidates(
        best_cfg=best_pre,
        grids={
            "edge_size": GRID_EDGE_SIZE,
            "edge_ent": GRID_EDGE_ENT,
            "node_feat_size": GRID_NODE_FEAT_SIZE,
            "node_feat_ent": GRID_NODE_FEAT_ENT,
        },
        seen_cfgs=seen_cfgs,
        budget=STAGE_D_BUDGET,
    )

    if stage_d_coeffs:
        stage_d_epochs = EPOCHS_SEARCH if STAGE_D_USE_SEARCH_EPOCHS else EPOCHS_FINAL
        stage_d_df = run_stage_grid(
            stage_name="D",
            coeff_dicts=stage_d_coeffs,
            epochs=stage_d_epochs,
            seeds=SEEDS,
            target_nodes=search_target_nodes,
            base_model=base_model,
            graph_cache=graph_cache,
            pred_cache=pred_cache,
            dataset=dataset,
            train_graph_indices=train_graph_indices,
            node_types=node_types_eff,
            sparsity=SPARSITY,
            mask_baseline_mode=MASK_BASELINE_MODE,
            ig_baseline_mode=IG_BASELINE_MODE,
            explanation_type=GNN_EXPLANATION_TYPE,
            threshold_cfg=THRESHOLD_CONFIG,
            topk_cfg=TOPK_CONFIG,
            median_cache=median_cache,
            elem_distribution_cache=elem_distribution_cache,
            baseline_cache_dir=baseline_cache_dir,
            lr=LR,
        )
        stage_d_summary = aggregate_stage_scores(stage_d_df)
        stage_d_top_df = select_top_k_configs(stage_d_summary, k=min(TOP_K_STAGE, max(1, STAGE_D_BUDGET)))

        raw_parts.append(stage_d_df)
        summary_parts.append(stage_d_summary)
    else:
        stage_d_df = pd.DataFrame()
        stage_d_summary = pd.DataFrame()
        stage_d_top_df = pd.DataFrame()
else:
    stage_d_df = pd.DataFrame()
    stage_d_summary = pd.DataFrame()
    stage_d_top_df = pd.DataFrame()

# Combined search summary and ranking
stage_summary_df = pd.concat(summary_parts, ignore_index=True) if summary_parts else pd.DataFrame()
search_summary_df = stage_summary_df[stage_summary_df["stage"].isin(["A", "B", "C", "D"])].copy() if not stage_summary_df.empty else pd.DataFrame()

if search_summary_df.empty:
    raise RuntimeError("Search stages produced no valid results.")

search_summary_df["fidelity_median"] = pd.to_numeric(search_summary_df["fidelity_median"], errors="coerce")
if not np.isfinite(search_summary_df["fidelity_median"].to_numpy()).any():
    diag = pd.concat(raw_parts, ignore_index=True) if raw_parts else pd.DataFrame()
    for c in ["fidelity_median", "n_nodes_evaluated", "n_nodes_requested", "n_failures", "error_top_count"]:
        if c in diag.columns:
            diag[c] = pd.to_numeric(diag[c], errors="coerce")
    if not diag.empty:
        diag_summary = (
            diag.groupby("stage", dropna=False)
            .agg(
                rows=("stage", "size"),
                finite_fidelity=("fidelity_median", lambda s: int(np.isfinite(s).sum())),
                median_nodes_eval=("n_nodes_evaluated", "median"),
                median_nodes_req=("n_nodes_requested", "median"),
                median_failures=("n_failures", "median"),
                median_error_top_count=("error_top_count", "median"),
            )
            .reset_index()
        )
        if "error_top_reason" in diag.columns:
            reason_summary = (
                diag[diag["error_top_reason"].astype(str) != ""]
                .groupby(["stage", "error_top_reason"], dropna=False)
                .size()
                .reset_index(name="rows")
                .sort_values(["stage", "rows"], ascending=[True, False])
            )
        else:
            reason_summary = pd.DataFrame()

        if "error_first_message" in diag.columns:
            msg_summary = (
                diag[diag["error_first_message"].astype(str) != ""]
                .groupby(["stage", "error_first_message"], dropna=False)
                .size()
                .reset_index(name="rows")
                .sort_values(["stage", "rows"], ascending=[True, False])
            )
        else:
            msg_summary = pd.DataFrame()

        print("Search diagnostics by stage:")
        display(diag_summary)
        if not reason_summary.empty:
            print("Top error reasons:")
            display(reason_summary.groupby("stage", dropna=False).head(5))
        if not msg_summary.empty:
            print("Top error messages:")
            display(msg_summary.groupby("stage", dropna=False).head(3))
    raise RuntimeError(
        "Search produced no finite fidelities. Check diagnostics; likely all explainer runs failed for selected nodes/config."
    )


best_search_df = select_top_k_configs(search_summary_df, k=1)
best_search_row = best_search_df.iloc[0]

best_config = {
    "edge_size": float(best_search_row["edge_size"]),
    "edge_ent": float(best_search_row["edge_ent"]),
    "node_feat_size": float(best_search_row["node_feat_size"]),
    "node_feat_ent": float(best_search_row["node_feat_ent"]),
    "lr": float(LR),
    "epochs": int(EPOCHS_FINAL),
}

# Final evaluation on FINAL_NODES with best config
final_coeffs = [
    {
        "edge_size": best_config["edge_size"],
        "edge_ent": best_config["edge_ent"],
        "node_feat_size": best_config["node_feat_size"],
        "node_feat_ent": best_config["node_feat_ent"],
    }
]

final_results_df = run_stage_grid(
    stage_name="FINAL",
    coeff_dicts=final_coeffs,
    epochs=EPOCHS_FINAL,
    seeds=SEEDS,
    target_nodes=final_target_nodes,
    base_model=base_model,
    graph_cache=graph_cache,
    pred_cache=pred_cache,
    dataset=dataset,
    train_graph_indices=train_graph_indices,
    node_types=node_types_eff,
    sparsity=SPARSITY,
    mask_baseline_mode=MASK_BASELINE_MODE,
    ig_baseline_mode=IG_BASELINE_MODE,
    explanation_type=GNN_EXPLANATION_TYPE,
    threshold_cfg=THRESHOLD_CONFIG,
    topk_cfg=TOPK_CONFIG,
    median_cache=median_cache,
    elem_distribution_cache=elem_distribution_cache,
    baseline_cache_dir=baseline_cache_dir,
    lr=LR,
)
final_summary_df = aggregate_stage_scores(final_results_df)

# Full outputs
grid_results_df = pd.concat(raw_parts + ([final_results_df] if not final_results_df.empty else []), ignore_index=True)
stage_summary_df = pd.concat([stage_summary_df, final_summary_df], ignore_index=True) if not final_summary_df.empty else stage_summary_df

pareto_df = compute_pareto_front(search_summary_df)

required_cols = [
    "stage",
    "edge_size",
    "edge_ent",
    "node_feat_size",
    "node_feat_ent",
    "lr",
    "epochs",
    "seed",
    "fidelity_median",
    "fidelity_iqr",
    "abs_delta_median",
    "edges_selected_median",
    "feats_selected_median",
    "runtime_sec",
]
for col in required_cols:
    if col not in grid_results_df.columns:
        grid_results_df[col] = np.nan

print("Best config (search -> final):")
print(best_config)
print("\nTop Stage A candidates:")
display(stage_a_top_df)
print("Top Stage B candidates:")
display(stage_b_top_df)
if not stage_c_summary.empty:
    print("Top Stage C candidates:")
    display(select_top_k_configs(stage_c_summary, k=TOP_K_STAGE))
if not stage_d_summary.empty:
    print("Top Stage D candidates:")
    display(stage_d_top_df)

print("\nFinal summary:")
display(final_summary_df)

if not pareto_df.empty:
    print("Pareto front (search stages):")
    display(pareto_df.head(15))

print("grid_results_df columns:")
print(list(grid_results_df.columns))







Candidate nodes: 201
Search nodes: 30
Final nodes: 150
[A] start: 126 runs


Stage A:   0%|          | 0/126 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Plotting: parallel coordinates + tradeoff scatters + diagnostics


def _build_plot_df_from_seed_rows(seed_df):
    if seed_df is None or seed_df.empty:
        return pd.DataFrame()

    group_cols = [
        "stage",
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "lr",
        "epochs",
    ]
    metric_cols = [
        "fidelity_median",
        "fidelity_iqr",
        "abs_delta_median",
        "runtime_sec",
        "n_nodes_evaluated",
        "n_nodes_requested",
        "n_failures",
    ]

    rows = []
    for keys, gdf in seed_df.groupby(group_cols, dropna=False):
        row = {col: val for col, val in zip(group_cols, keys)}
        for metric in metric_cols:
            vals = pd.to_numeric(gdf[metric], errors="coerce").dropna()
            row[metric] = float(vals.median()) if not vals.empty else float("nan")
        rows.append(row)

    return pd.DataFrame(rows)


def _coerce_numeric(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def _finite_rows(df, cols):
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()

    mask = np.ones(len(df), dtype=bool)
    for col in cols:
        if col not in df.columns:
            return pd.DataFrame(columns=df.columns)
        mask &= np.isfinite(pd.to_numeric(df[col], errors="coerce").to_numpy())
    return df.loc[mask].copy()


def _missing_cols(df, required):
    return [col for col in required if col not in df.columns]


plot_df = pd.DataFrame()

# Preferred source: aggregated stage summary
if "stage_summary_df" in globals() and isinstance(stage_summary_df, pd.DataFrame) and not stage_summary_df.empty:
    plot_df = stage_summary_df.copy()

# Fallback: aggregate from seed-level rows
if plot_df.empty and "grid_results_df" in globals() and isinstance(grid_results_df, pd.DataFrame) and not grid_results_df.empty:
    plot_df = _build_plot_df_from_seed_rows(grid_results_df.copy())

if plot_df.empty:
    raise RuntimeError("No plot data found. Run the staged gridsearch cell first.")

plot_df = _coerce_numeric(
    plot_df,
    [
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "fidelity_median",
        "fidelity_iqr",
        "abs_delta_median",
        "runtime_sec",
    ],
)

plot_df = _finite_rows(plot_df, ["fidelity_median"])
if plot_df.empty:
    print("No finite fidelity values found for plotting.")

    if "grid_results_df" in globals() and isinstance(grid_results_df, pd.DataFrame) and not grid_results_df.empty:
        diag = grid_results_df.copy()
        for c in ["fidelity_median", "n_nodes_evaluated", "n_nodes_requested", "n_failures"]:
            if c in diag.columns:
                diag[c] = pd.to_numeric(diag[c], errors="coerce")

        diag_summary = (
            diag.groupby("stage", dropna=False)
            .agg(
                rows=("stage", "size"),
                finite_fidelity=("fidelity_median", lambda s: int(np.isfinite(s).sum())),
                median_nodes_eval=("n_nodes_evaluated", "median"),
                median_nodes_req=("n_nodes_requested", "median"),
                median_failures=("n_failures", "median"),
            )
            .reset_index()
        )
        print("Diagnostic summary by stage:")
        display(diag_summary)

    raise RuntimeError(
        "No finite values available for plotting. Inspect diagnostic summary above and rerun search cell."
    )

for col in ["edge_size", "edge_ent", "node_feat_size", "node_feat_ent"]:
    if col not in plot_df.columns:
        print(f"Skip log transform for {col}: column missing.")
        continue
    vals = pd.to_numeric(plot_df[col], errors="coerce").astype(float)
    plot_df[f"log10_{col}"] = np.log10(np.clip(vals, 1e-12, None))

parallel_dims = [
    "log10_edge_size",
    "log10_edge_ent",
    "log10_node_feat_size",
    "log10_node_feat_ent",
    "fidelity_median",
    "fidelity_iqr",
    "abs_delta_median",
    "runtime_sec",
]

missing_parallel = _missing_cols(plot_df, parallel_dims)
if missing_parallel:
    print(f"Skip parallel coordinates: missing columns {missing_parallel}")
else:
    parallel_df = _finite_rows(plot_df, parallel_dims)
    if parallel_df.empty:
        print("Skip parallel coordinates: no finite rows for all selected dimensions.")
    else:
        fig_parallel = px.parallel_coordinates(
            parallel_df,
            dimensions=parallel_dims,
            color="fidelity_median",
            color_continuous_scale=px.colors.sequential.Viridis,
            labels={
                "log10_edge_size": "log10(edge_size)",
                "log10_edge_ent": "log10(edge_ent)",
                "log10_node_feat_size": "log10(node_feat_size)",
                "log10_node_feat_ent": "log10(node_feat_ent)",
                "fidelity_median": "fidelity_median",
                "fidelity_iqr": "fidelity_iqr",
                "abs_delta_median": "abs_delta_median",
                "runtime_sec": "runtime_sec",
            },
        )
        # fig_parallel.update_layout(title="GNNExplainer Hyperparameter Gridsearch - Parallel Coordinates")
        fig_parallel.show()

scatter_hover_cols = [
    col
    for col in [
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "fidelity_iqr",
        "abs_delta_median",
        "runtime_sec",
    ]
    if col in plot_df.columns
]

runtime_required = ["stage", "runtime_sec", "fidelity_median"]
if _missing_cols(plot_df, runtime_required):
    print(f"Skip runtime tradeoff scatter: missing columns {_missing_cols(plot_df, runtime_required)}")
else:
    runtime_df = _finite_rows(plot_df, ["runtime_sec", "fidelity_median"])
    if runtime_df.empty:
        print("Skip runtime tradeoff scatter: no finite points.")
    else:
        fig_runtime = px.scatter(
            runtime_df,
            x="runtime_sec",
            y="fidelity_median",
            color="stage",
            hover_data=scatter_hover_cols,
            title="Tradeoff: fidelity_median vs runtime_sec",
        )
        fig_runtime.show()

abs_required = ["stage", "abs_delta_median", "fidelity_median"]
if _missing_cols(plot_df, abs_required):
    print(f"Skip abs-delta tradeoff scatter: missing columns {_missing_cols(plot_df, abs_required)}")
else:
    abs_df = _finite_rows(plot_df, ["abs_delta_median", "fidelity_median"])
    if abs_df.empty:
        print("Skip abs-delta tradeoff scatter: no finite points.")
    else:
        fig_abs_delta = px.scatter(
            abs_df,
            x="abs_delta_median",
            y="fidelity_median",
            color="stage",
            hover_data=scatter_hover_cols,
            title="Tradeoff: fidelity_median vs abs_delta_median",
        )
        fig_abs_delta.show()

stage_source = pd.DataFrame()
if "stage_summary_df" in globals() and isinstance(stage_summary_df, pd.DataFrame) and not stage_summary_df.empty:
    stage_source = stage_summary_df.copy()
elif not plot_df.empty:
    stage_source = plot_df.copy()

if stage_source.empty:
    print("Skip stage heatmaps: stage summary data not available.")
else:
    stage_source = _coerce_numeric(
        stage_source,
        [
            "edge_size",
            "edge_ent",
            "node_feat_size",
            "node_feat_ent",
            "fidelity_median",
        ],
    )

    # Stage A heatmap
    stage_a_required = ["stage", "edge_size", "edge_ent", "fidelity_median"]
    if _missing_cols(stage_source, stage_a_required):
        print(f"Skip Stage A heatmap: missing columns {_missing_cols(stage_source, stage_a_required)}")
    else:
        stage_a_df = stage_source.loc[stage_source["stage"].astype(str) == "A"].copy()
        stage_a_df = _finite_rows(stage_a_df, ["edge_size", "edge_ent", "fidelity_median"])
        if stage_a_df.empty:
            print("Skip Stage A heatmap: no finite Stage A rows.")
        else:
            stage_a_pivot = (
                stage_a_df.pivot_table(
                    index="edge_ent",
                    columns="edge_size",
                    values="fidelity_median",
                    aggfunc="median",
                )
                .sort_index()
                .sort_index(axis=1)
            )
            if stage_a_pivot.empty:
                print("Skip Stage A heatmap: pivot table empty.")
            else:
                fig_heat_a = px.imshow(
                    stage_a_pivot,
                    origin="lower",
                    aspect="auto",
                    color_continuous_scale=px.colors.sequential.Viridis,
                    labels={"x": "edge_size", "y": "edge_ent", "color": "fidelity_median"},
                    title="Stage A heatmap: fidelity_median by edge_size x edge_ent",
                )
                fig_heat_a.show()

    # Stage B heatmap
    stage_b_required = ["stage", "node_feat_size", "node_feat_ent", "fidelity_median"]
    if _missing_cols(stage_source, stage_b_required):
        print(f"Skip Stage B heatmap: missing columns {_missing_cols(stage_source, stage_b_required)}")
    else:
        stage_b_df = stage_source.loc[stage_source["stage"].astype(str) == "B"].copy()
        stage_b_df = _finite_rows(stage_b_df, ["node_feat_size", "node_feat_ent", "fidelity_median"])
        if stage_b_df.empty:
            print("Skip Stage B heatmap: no finite Stage B rows.")
        else:
            stage_b_pivot = (
                stage_b_df.pivot_table(
                    index="node_feat_ent",
                    columns="node_feat_size",
                    values="fidelity_median",
                    aggfunc="median",
                )
                .sort_index()
                .sort_index(axis=1)
            )
            if stage_b_pivot.empty:
                print("Skip Stage B heatmap: pivot table empty.")
            else:
                fig_heat_b = px.imshow(
                    stage_b_pivot,
                    origin="lower",
                    aspect="auto",
                    color_continuous_scale=px.colors.sequential.Viridis,
                    labels={"x": "node_feat_size", "y": "node_feat_ent", "color": "fidelity_median"},
                    title="Stage B heatmap: fidelity_median by node_feat_size x node_feat_ent",
                )
                fig_heat_b.show()

# Pareto scatter on search stages
if "search_summary_df" in globals() and isinstance(search_summary_df, pd.DataFrame) and not search_summary_df.empty:
    pareto_base = search_summary_df.copy()
    pareto_base = _coerce_numeric(
        pareto_base,
        [
            "edge_size",
            "edge_ent",
            "node_feat_size",
            "node_feat_ent",
            "runtime_sec",
            "fidelity_median",
            "abs_delta_median",
            "fidelity_iqr",
            "fidelity_std",
        ],
    )

    pareto_required = [
        "stage",
        "edge_size",
        "edge_ent",
        "node_feat_size",
        "node_feat_ent",
        "runtime_sec",
        "fidelity_median",
    ]
    missing_pareto = _missing_cols(pareto_base, pareto_required)
    if missing_pareto:
        print(f"Skip Pareto scatter: missing columns {missing_pareto}")
    else:
        pareto_base = _finite_rows(pareto_base, ["runtime_sec", "fidelity_median"])
        if pareto_base.empty:
            print("Skip Pareto scatter: no finite points.")
        else:
            if "pareto_df" in globals() and isinstance(pareto_df, pd.DataFrame) and not pareto_df.empty:
                pareto_ref = pareto_df.copy()
            elif "compute_pareto_front" in globals():
                pareto_ref = compute_pareto_front(pareto_base)
            else:
                pareto_ref = pd.DataFrame()

            key_cols = ["stage", "edge_size", "edge_ent", "node_feat_size", "node_feat_ent"]
            if pareto_ref.empty or _missing_cols(pareto_ref, key_cols):
                pareto_plot_df = pareto_base.copy()
                pareto_plot_df["pareto_status"] = "Unknown"
                print("Pareto reference unavailable: marking points as Unknown.")
            else:
                pareto_keys = pareto_ref[key_cols].drop_duplicates().copy()
                pareto_keys["pareto_status"] = "Pareto"
                pareto_plot_df = pareto_base.merge(pareto_keys, on=key_cols, how="left")
                pareto_plot_df["pareto_status"] = pareto_plot_df["pareto_status"].fillna("Dominated")

            pareto_hover_cols = [
                col
                for col in [
                    "edge_size",
                    "edge_ent",
                    "node_feat_size",
                    "node_feat_ent",
                    "runtime_sec",
                    "fidelity_iqr",
                    "abs_delta_median",
                    "fidelity_std",
                ]
                if col in pareto_plot_df.columns
            ]

            fig_pareto = px.scatter(
                pareto_plot_df,
                x="runtime_sec",
                y="fidelity_median",
                color="pareto_status",
                symbol="stage",
                hover_data=pareto_hover_cols,
                color_discrete_map={
                    "Pareto": "#1f77b4",
                    "Dominated": "#7f7f7f",
                    "Unknown": "#bcbd22",
                },
                title="Pareto: fidelity_median vs runtime_sec (search stages)",
            )
            fig_pareto.show()
else:
    print("Skip Pareto scatter: search_summary_df is not available.")

# Seed stability plot: fidelity_median +/- fidelity_std by stage
seed_source = pd.DataFrame()
if "search_summary_df" in globals() and isinstance(search_summary_df, pd.DataFrame) and not search_summary_df.empty:
    seed_source = search_summary_df.copy()
elif "stage_summary_df" in globals() and isinstance(stage_summary_df, pd.DataFrame) and not stage_summary_df.empty:
    seed_source = stage_summary_df.copy()

seed_required = [
    "stage",
    "edge_size",
    "edge_ent",
    "node_feat_size",
    "node_feat_ent",
    "fidelity_median",
    "fidelity_std",
]

if seed_source.empty:
    print("Skip seed stability plot: no summary data available.")
elif _missing_cols(seed_source, seed_required):
    print(f"Skip seed stability plot: missing columns {_missing_cols(seed_source, seed_required)}")
else:
    seed_plot_df = _coerce_numeric(
        seed_source,
        [
            "edge_size",
            "edge_ent",
            "node_feat_size",
            "node_feat_ent",
            "fidelity_median",
            "fidelity_std",
        ],
    )
    seed_plot_df = _finite_rows(seed_plot_df, ["fidelity_median"])
    if seed_plot_df.empty:
        print("Skip seed stability plot: no finite fidelity values.")
    else:
        seed_plot_df["fidelity_std"] = pd.to_numeric(seed_plot_df["fidelity_std"], errors="coerce").fillna(0.0)
        seed_plot_df["config_id"] = seed_plot_df.apply(
            lambda row: (
                f"es={float(row['edge_size']):.1e} | "
                f"ee={float(row['edge_ent']):.1e} | "
                f"nfs={float(row['node_feat_size']):.1e} | "
                f"nfe={float(row['node_feat_ent']):.1e}"
            ),
            axis=1,
        )
        seed_plot_df = seed_plot_df.sort_values(["stage", "fidelity_median"], ascending=[True, False])

        seed_hover_cols = [
            col
            for col in [
                "edge_size",
                "edge_ent",
                "node_feat_size",
                "node_feat_ent",
                "fidelity_std",
                "runtime_sec",
                "abs_delta_median",
            ]
            if col in seed_plot_df.columns
        ]

        fig_seed = px.scatter(
            seed_plot_df,
            x="config_id",
            y="fidelity_median",
            error_y="fidelity_std",
            color="stage",
            facet_col="stage",
            facet_col_wrap=2,
            hover_data=seed_hover_cols,
            title="Seed stability: fidelity_median +/- fidelity_std by stage",
        )
        fig_seed.update_xaxes(tickangle=45)
        fig_seed.show()


### Variability Gridsearch for GNNExplainer

Dedicated staged gridsearch (A/B/C/(D)+FINAL) optimized for low explanation variability across repeated seeds on the same input nodes/graphs.
Primary objective: minimize `Mask Value Var` (feature + edge), tie-break by higher fidelity.
Sparsity is fixed to `0.80`.

<!-- VARIABILITY_GRIDSEARCH_SECTION_V1 -->

In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Config
VAR_SEARCH_NODES = int(globals().get("SEARCH_NODES", 30))
VAR_FINAL_NODES = int(globals().get("FINAL_NODES", 150))
VAR_SEEDS = [0, 1, 2, 3, 4]
VAR_SPARSITY = 0.80

VAR_TOP_K_STAGE = int(globals().get("TOP_K_STAGE", 3))
VAR_STAGE_D_BUDGET = 0
VAR_STAGE_D_USE_SEARCH_EPOCHS = True
VAR_MIN_SEED_RUNS_PER_NODE = 2

VAR_EPOCHS_SEARCH = int(globals().get("EPOCHS_SEARCH", 150))
VAR_EPOCHS_FINAL = int(globals().get("EPOCHS_FINAL", 250))
VAR_LR = float(globals().get("LR", 0.01))

print("Variability gridsearch config loaded.")
print(f"VAR_SEARCH_NODES={VAR_SEARCH_NODES}, VAR_FINAL_NODES={VAR_FINAL_NODES}")
print(f"VAR_SEEDS={VAR_SEEDS}")
print(f"VAR_SPARSITY={VAR_SPARSITY:.2f}")


Variability gridsearch config loaded.
VAR_SEARCH_NODES=30, VAR_FINAL_NODES=150
VAR_SEEDS=[0, 1, 2, 3, 4]
VAR_SPARSITY=0.80


In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Helpers (mask collection)
from datetime import datetime
import json

VAR_GROUP_COLS = [
    "stage", "edge_size", "edge_ent", "node_feat_size", "node_feat_ent", "lr", "epochs"
]


def var_sparse_topk(values, sparsity):
    arr = np.asarray(values, dtype=float).reshape(-1)
    if arr.size == 0:
        return np.asarray([], dtype=float)
    keep_ratio = min(max(1.0 - float(sparsity), 0.0), 1.0)
    k_keep = max(1, int(math.ceil(keep_ratio * int(arr.size))))
    k_keep = min(k_keep, int(arr.size))
    order = np.argsort(-np.where(np.isfinite(np.abs(arr)), np.abs(arr), -np.inf))
    sparse = np.zeros_like(arr, dtype=float)
    sel = order[:k_keep]
    sparse[sel] = np.where(np.isfinite(arr[sel]), arr[sel], 0.0)
    return sparse


def var_incident_entries(edge_index_dict, node_type, node_idx):
    node_idx = int(node_idx)
    out = []
    for edge_type, edge_index in edge_index_dict.items():
        if edge_index is None:
            continue
        src_type, _, dst_type = edge_type
        src = edge_index[0].detach().cpu().numpy().astype(int)
        dst = edge_index[1].detach().cpu().numpy().astype(int)
        if src_type == node_type and dst_type == node_type:
            mask = (src == node_idx) | (dst == node_idx)
        elif src_type == node_type:
            mask = src == node_idx
        elif dst_type == node_type:
            mask = dst == node_idx
        else:
            continue
        for pos in np.where(mask)[0].tolist():
            out.append((edge_type, int(pos)))
    return sorted(set(out), key=lambda t: (str(t[0][0]), str(t[0][1]), str(t[0][2]), int(t[1])))


def build_incident_edge_cache(graph_cache, target_nodes):
    cache = {}
    for row in target_nodes:
        key = (int(row["graph_idx"]), str(row["node_type"]), int(row["node_idx"]))
        if key in cache:
            continue
        graph_idx, node_type, node_idx = key
        if graph_idx not in graph_cache:
            cache[key] = []
            continue
        edge_index_dict = graph_cache[graph_idx]["edge_index_dict"]
        cache[key] = var_incident_entries(edge_index_dict, node_type=node_type, node_idx=node_idx)
    return cache


def var_edge_sparse_mask(edge_mask_dict, incident_entries, sparsity):
    if not incident_entries:
        return np.asarray([], dtype=float)
    dense = np.zeros(len(incident_entries), dtype=float)
    if edge_mask_dict is None:
        return dense
    for i, (edge_type, edge_pos) in enumerate(incident_entries):
        edge_mask = edge_mask_dict.get(edge_type)
        if edge_mask is None:
            continue
        flat = edge_mask.view(-1).detach().cpu().numpy().astype(float)
        if 0 <= int(edge_pos) < int(flat.size):
            v = float(flat[int(edge_pos)])
            dense[i] = v if np.isfinite(v) else 0.0
    return var_sparse_topk(dense, sparsity=sparsity)


def collect_mask_seed_rows_for_stage(
    stage_name,
    coeff_dicts,
    epochs,
    seeds,
    target_nodes,
    base_model,
    graph_cache,
    node_types,
    explanation_type,
    sparsity,
    incident_edge_cache,
    lr,
):
    rows = []
    total_runs = int(len(coeff_dicts) * len(seeds))
    run_iter = itertools.product(coeff_dicts, seeds)
    run_iter = safe_tqdm(
        run_iter,
        total=total_runs,
        desc=f"VarMask Stage {stage_name}",
        enabled=bool(globals().get("PROGRESS_ENABLED", True)) and str(globals().get("PROGRESS_GRANULARITY", "stage_config_seed")) == "stage_config_seed",
        use_tqdm=bool(globals().get("PROGRESS_USE_TQDM", True)),
    )

    for coeffs, seed in run_iter:
        set_all_seeds(seed)
        explainers = {}
        model_cfg = ModelConfig(
            mode=ModelMode.regression,
            task_level=ModelTaskLevel.node,
            return_type=ModelReturnType.raw,
        )
        for nt in node_types:
            explainers[nt] = Explainer(
                model=NodeTypeRegressionWrapper(base_model, nt),
                algorithm=build_gnnexplainer_algorithm(epochs=epochs, lr=lr, coeffs=coeffs),
                explanation_type=str(explanation_type),
                model_config=model_cfg,
                node_mask_type="attributes",
                edge_mask_type="object",
            )

        for row in target_nodes:
            graph_idx = int(row["graph_idx"])
            node_type = str(row["node_type"])
            node_idx = int(row["node_idx"])
            target_value = float(row.get("target_value", float("nan")))
            if node_type not in explainers or graph_idx not in graph_cache:
                continue

            cache_item = graph_cache[graph_idx]
            x_dict = cache_item["x_dict"]
            edge_index_dict = cache_item["edge_index_dict"]
            edge_attr_dict = cache_item["edge_attr_dict"]
            y_dict = cache_item["y_dict"]

            gnn_target = None
            if str(explanation_type) == "phenomenon":
                y_tensor = y_dict.get(node_type)
                if y_tensor is None:
                    continue
                y_tensor = y_tensor.reshape(-1)
                if node_idx >= int(y_tensor.size(0)) or torch.isnan(y_tensor[node_idx]):
                    continue
                gnn_target = y_tensor

            try:
                try:
                    explanation = explainers[node_type](
                        {nt: feat.clone() for nt, feat in x_dict.items()},
                        edge_index_dict,
                        edge_attr_dict=_clone_edge_attr_dict(edge_attr_dict),
                        target=gnn_target,
                        index=node_idx,
                    )
                except Exception as exc_explain:
                    if str(explanation_type) == "phenomenon" and gnn_target is not None:
                        explanation = explainers[node_type](
                            {nt: feat.clone() for nt, feat in x_dict.items()},
                            edge_index_dict,
                            edge_attr_dict=_clone_edge_attr_dict(edge_attr_dict),
                            target=gnn_target[node_idx],
                            index=node_idx,
                        )
                    else:
                        raise exc_explain

                feat_vals = exp_extract_gnn_feature_importance(explanation, node_type=node_type, node_idx=node_idx)
                feat_sparse = var_sparse_topk(feat_vals, sparsity=sparsity)
                incident = incident_edge_cache.get((graph_idx, node_type, node_idx), [])
                edge_sparse = var_edge_sparse_mask(explanation.edge_mask_dict, incident_entries=incident, sparsity=sparsity)

                rows.append(
                    {
                        "stage": str(stage_name),
                        "edge_size": float(coeffs["edge_size"]),
                        "edge_ent": float(coeffs["edge_ent"]),
                        "node_feat_size": float(coeffs["node_feat_size"]),
                        "node_feat_ent": float(coeffs["node_feat_ent"]),
                        "lr": float(lr),
                        "epochs": int(epochs),
                        "seed": int(seed),
                        "graph_idx": int(graph_idx),
                        "node_type": str(node_type),
                        "node_idx": int(node_idx),
                        "target_value": float(target_value),
                        "feature_sparse_mask": feat_sparse,
                        "edge_sparse_mask": edge_sparse,
                    }
                )
            except Exception:
                continue

    return pd.DataFrame(rows)


In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Helpers (aggregation + ranking)

def var_mean_dim_variance(vectors):
    if len(vectors) < 2:
        return float("nan")
    arrs = [np.asarray(v, dtype=float).reshape(-1) for v in vectors]
    max_dim = max((int(a.size) for a in arrs), default=0)
    if max_dim <= 0:
        return float("nan")
    mat = np.zeros((len(arrs), max_dim), dtype=float)
    for i, arr in enumerate(arrs):
        if arr.size > 0:
            mat[i, : int(arr.size)] = arr
    return float(np.var(mat, axis=0, ddof=0).mean())


def aggregate_var_nodes(mask_seed_df, min_seed_runs=2):
    if mask_seed_df is None or mask_seed_df.empty:
        return pd.DataFrame()
    group_cols = VAR_GROUP_COLS + ["graph_idx", "node_type", "node_idx"]
    rows = []
    for keys, gdf in mask_seed_df.groupby(group_cols, dropna=False):
        seed_runs = int(pd.to_numeric(gdf["seed"], errors="coerce").nunique())
        if seed_runs < int(min_seed_runs):
            continue
        feat_var = var_mean_dim_variance(gdf["feature_sparse_mask"].tolist())
        edge_var = var_mean_dim_variance(gdf["edge_sparse_mask"].tolist())
        finite = [v for v in [feat_var, edge_var] if np.isfinite(v)]
        mask_var = float(np.mean(finite)) if finite else float("nan")
        row = {col: val for col, val in zip(group_cols, keys)}
        tgt = pd.to_numeric(gdf["target_value"], errors="coerce").dropna()
        row.update(
            {
                "target_value": float(tgt.median()) if not tgt.empty else float("nan"),
                "seed_runs": seed_runs,
                "feature_mask_var_mean": float(feat_var),
                "edge_mask_var_mean": float(edge_var),
                "mask_var_mean": float(mask_var),
            }
        )
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    out = pd.DataFrame(rows)
    return out.sort_values(["stage", "mask_var_mean"], ascending=[True, True]).reset_index(drop=True)


def aggregate_var_stage(var_node_df):
    if var_node_df is None or var_node_df.empty:
        return pd.DataFrame()
    rows = []
    for keys, gdf in var_node_df.groupby(VAR_GROUP_COLS, dropna=False):
        row = {col: val for col, val in zip(VAR_GROUP_COLS, keys)}
        mask_vals = pd.to_numeric(gdf["mask_var_mean"], errors="coerce").dropna()
        feat_vals = pd.to_numeric(gdf["feature_mask_var_mean"], errors="coerce").dropna()
        edge_vals = pd.to_numeric(gdf["edge_mask_var_mean"], errors="coerce").dropna()
        seed_runs_vals = pd.to_numeric(gdf["seed_runs"], errors="coerce").dropna()
        row.update(
            {
                "nodes_with_variability": int(mask_vals.shape[0]),
                "mask_var_median": float(mask_vals.median()) if not mask_vals.empty else float("nan"),
                "mask_var_iqr": _iqr(mask_vals.to_numpy(dtype=float)) if not mask_vals.empty else float("nan"),
                "mask_var_std": float(mask_vals.std(ddof=0)) if not mask_vals.empty else float("nan"),
                "feature_mask_var_median": float(feat_vals.median()) if not feat_vals.empty else float("nan"),
                "edge_mask_var_median": float(edge_vals.median()) if not edge_vals.empty else float("nan"),
                "node_seed_runs_min": float(seed_runs_vals.min()) if not seed_runs_vals.empty else float("nan"),
            }
        )
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)


def build_var_stage_summary(fid_seed_df, var_node_df):
    fid_summary = aggregate_stage_scores(fid_seed_df)
    var_summary = aggregate_var_stage(var_node_df)
    if fid_summary.empty:
        return pd.DataFrame()
    if var_summary.empty:
        for col in ["nodes_with_variability", "mask_var_median", "mask_var_iqr", "mask_var_std", "feature_mask_var_median", "edge_mask_var_median", "node_seed_runs_min"]:
            fid_summary[col] = np.nan
        return fid_summary
    merged = fid_summary.merge(var_summary, on=VAR_GROUP_COLS, how="left")
    return merged


def select_top_k_variability_configs(df_stage_agg, k=3):
    if df_stage_agg is None or df_stage_agg.empty:
        return pd.DataFrame()
    work = df_stage_agg.copy()
    for col in ["mask_var_median", "fidelity_median", "selection_sum", "runtime_sec"]:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work = work[np.isfinite(work["mask_var_median"])].copy()
    if work.empty:
        return pd.DataFrame()
    work = work.sort_values(
        ["mask_var_median", "fidelity_median", "selection_sum", "runtime_sec"],
        ascending=[True, False, True, True],
        na_position="last",
    )
    return work.head(max(0, int(k))).reset_index(drop=True)


In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Stage runner helper

def run_var_stage(
    stage_name,
    coeff_dicts,
    epochs,
    seeds,
    target_nodes,
    base_model,
    graph_cache,
    pred_cache,
    dataset,
    train_graph_indices,
    node_types,
    incident_edge_cache,
    lr,
):
    fid_seed_df = run_stage_grid(
        stage_name=stage_name,
        coeff_dicts=coeff_dicts,
        epochs=epochs,
        seeds=seeds,
        target_nodes=target_nodes,
        base_model=base_model,
        graph_cache=graph_cache,
        pred_cache=pred_cache,
        dataset=dataset,
        train_graph_indices=train_graph_indices,
        node_types=node_types,
        sparsity=VAR_SPARSITY,
        mask_baseline_mode=MASK_BASELINE_MODE,
        ig_baseline_mode=IG_BASELINE_MODE,
        explanation_type=GNN_EXPLANATION_TYPE,
        threshold_cfg=THRESHOLD_CONFIG,
        topk_cfg=TOPK_CONFIG,
        median_cache=var_median_cache,
        elem_distribution_cache=var_elem_distribution_cache,
        baseline_cache_dir=var_baseline_cache_dir,
        lr=lr,
    )

    mask_seed_df = collect_mask_seed_rows_for_stage(
        stage_name=stage_name,
        coeff_dicts=coeff_dicts,
        epochs=epochs,
        seeds=seeds,
        target_nodes=target_nodes,
        base_model=base_model,
        graph_cache=graph_cache,
        node_types=node_types,
        explanation_type=GNN_EXPLANATION_TYPE,
        sparsity=VAR_SPARSITY,
        incident_edge_cache=incident_edge_cache,
        lr=lr,
    )

    var_node_df = aggregate_var_nodes(mask_seed_df, min_seed_runs=VAR_MIN_SEED_RUNS_PER_NODE)
    stage_summary_df = build_var_stage_summary(fid_seed_df, var_node_df)
    return fid_seed_df, mask_seed_df, var_node_df, stage_summary_df


In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Run search stages A/B/C/(D)
set_all_seeds(VAR_SEEDS[0])

var_artifacts = load_eval_artifacts(
    project_root=PROJECT_ROOT,
    model_file=MODEL_FILE,
    data_file=DATA_FILE,
    split_file=SPLIT_FILE,
    node_types=NODE_TYPES,
    graph_scope=GRAPH_SCOPE,
    first_graph_per_component=FIRST_GRAPH_PER_COMPONENT,
    component_key=COMPONENT_KEY,
)

var_dataset = var_artifacts["dataset"]
var_base_model = var_artifacts["base_model"]
var_device = var_artifacts["device"]
var_eval_graph_indices = var_artifacts["eval_graph_indices"]
var_train_graph_indices = var_artifacts["train_graph_indices"]
var_node_types_eff = var_artifacts["node_types"]
var_context = var_artifacts["context"]

var_candidate_nodes = collect_candidate_nodes(
    dataset=var_dataset,
    eval_graph_indices=var_eval_graph_indices,
    node_types=var_node_types_eff,
    explanation_type=GNN_EXPLANATION_TYPE,
)
if not var_candidate_nodes:
    raise RuntimeError("No valid candidate nodes found for variability search.")

var_search_target_nodes, var_final_target_nodes = sample_target_nodes(
    candidates=var_candidate_nodes,
    search_nodes=VAR_SEARCH_NODES,
    final_nodes=VAR_FINAL_NODES,
    seed=VAR_SEEDS[0],
)

var_search_keys = {(int(r["graph_idx"]), str(r["node_type"]), int(r["node_idx"])) for r in var_search_target_nodes}
var_all_target_nodes = list(var_search_target_nodes)
for r in var_final_target_nodes:
    key = (int(r["graph_idx"]), str(r["node_type"]), int(r["node_idx"]))
    if key not in var_search_keys:
        var_all_target_nodes.append(r)

var_graph_cache, var_pred_cache = build_base_prediction_cache(
    base_model=var_base_model,
    dataset=var_dataset,
    target_nodes=var_all_target_nodes,
    device=var_device,
)

var_median_cache = {}
var_elem_distribution_cache = {}
var_baseline_cache_dir = var_context.output_dir.parent / "baselines"
if str(IG_BASELINE_MODE) == "scientific":
    try:
        from scripts.explainer.baselines import load_or_compute_element_distribution, load_or_compute_medians
        for nt in var_node_types_eff:
            try:
                var_median_cache[nt] = load_or_compute_medians(
                    dataset=var_dataset,
                    train_graph_indices=var_train_graph_indices,
                    node_type=nt,
                    cache_dir=str(var_baseline_cache_dir),
                )
            except Exception:
                pass
        try:
            var_elem_distribution_cache["value"] = load_or_compute_element_distribution(
                dataset=var_dataset,
                train_graph_indices=var_train_graph_indices,
                cache_dir=str(var_baseline_cache_dir),
            )
        except Exception:
            pass
    except Exception:
        pass

var_incident_edge_cache = build_incident_edge_cache(var_graph_cache, var_all_target_nodes)

print(f"Variability candidate nodes: {len(var_candidate_nodes)}")
print(f"Variability search nodes: {len(var_search_target_nodes)}")
print(f"Variability final nodes: {len(var_final_target_nodes)}")

var_stage_a_coeffs = [
    {
        "edge_size": float(edge_size),
        "edge_ent": float(edge_ent),
        "node_feat_size": float(NEUTRAL_NODE_FEAT_SIZE),
        "node_feat_ent": float(NEUTRAL_NODE_FEAT_ENT),
    }
    for edge_size, edge_ent in itertools.product(GRID_EDGE_SIZE, GRID_EDGE_ENT)
]

var_stage_a_seed_df, var_stage_a_mask_seed_df, var_stage_a_node_df, var_stage_a_summary = run_var_stage(
    stage_name="A",
    coeff_dicts=var_stage_a_coeffs,
    epochs=VAR_EPOCHS_SEARCH,
    seeds=VAR_SEEDS,
    target_nodes=var_search_target_nodes,
    base_model=var_base_model,
    graph_cache=var_graph_cache,
    pred_cache=var_pred_cache,
    dataset=var_dataset,
    train_graph_indices=var_train_graph_indices,
    node_types=var_node_types_eff,
    incident_edge_cache=var_incident_edge_cache,
    lr=VAR_LR,
)

var_stage_b_coeffs = [
    {
        "edge_size": float(NEUTRAL_EDGE_SIZE),
        "edge_ent": float(NEUTRAL_EDGE_ENT),
        "node_feat_size": float(node_feat_size),
        "node_feat_ent": float(node_feat_ent),
    }
    for node_feat_size, node_feat_ent in itertools.product(GRID_NODE_FEAT_SIZE, GRID_NODE_FEAT_ENT)
]

var_stage_b_seed_df, var_stage_b_mask_seed_df, var_stage_b_node_df, var_stage_b_summary = run_var_stage(
    stage_name="B",
    coeff_dicts=var_stage_b_coeffs,
    epochs=VAR_EPOCHS_SEARCH,
    seeds=VAR_SEEDS,
    target_nodes=var_search_target_nodes,
    base_model=var_base_model,
    graph_cache=var_graph_cache,
    pred_cache=var_pred_cache,
    dataset=var_dataset,
    train_graph_indices=var_train_graph_indices,
    node_types=var_node_types_eff,
    incident_edge_cache=var_incident_edge_cache,
    lr=VAR_LR,
)

var_stage_c_coeffs = build_stage_c_candidates(
    select_top_k_variability_configs(var_stage_a_summary, k=VAR_TOP_K_STAGE),
    select_top_k_variability_configs(var_stage_b_summary, k=VAR_TOP_K_STAGE),
)
if var_stage_c_coeffs:
    var_stage_c_seed_df, var_stage_c_mask_seed_df, var_stage_c_node_df, var_stage_c_summary = run_var_stage(
        stage_name="C",
        coeff_dicts=var_stage_c_coeffs,
        epochs=VAR_EPOCHS_SEARCH,
        seeds=VAR_SEEDS,
        target_nodes=var_search_target_nodes,
        base_model=var_base_model,
        graph_cache=var_graph_cache,
        pred_cache=var_pred_cache,
        dataset=var_dataset,
        train_graph_indices=var_train_graph_indices,
        node_types=var_node_types_eff,
        incident_edge_cache=var_incident_edge_cache,
        lr=VAR_LR,
    )
else:
    var_stage_c_seed_df = pd.DataFrame(); var_stage_c_mask_seed_df = pd.DataFrame(); var_stage_c_node_df = pd.DataFrame(); var_stage_c_summary = pd.DataFrame()

var_seed_parts = [df for df in [var_stage_a_seed_df, var_stage_b_seed_df, var_stage_c_seed_df] if not df.empty]
var_mask_seed_parts = [df for df in [var_stage_a_mask_seed_df, var_stage_b_mask_seed_df, var_stage_c_mask_seed_df] if not df.empty]
var_node_parts = [df for df in [var_stage_a_node_df, var_stage_b_node_df, var_stage_c_node_df] if not df.empty]
var_summary_parts = [df for df in [var_stage_a_summary, var_stage_b_summary, var_stage_c_summary] if not df.empty]

var_search_summary_df = pd.concat(var_summary_parts, ignore_index=True) if var_summary_parts else pd.DataFrame()

if VAR_STAGE_D_BUDGET > 0 and not var_search_summary_df.empty:
    best_pre = select_top_k_variability_configs(var_search_summary_df, k=1).iloc[0].to_dict()
    seen_cfgs = {
        _coeff_signature({"edge_size": r.edge_size, "edge_ent": r.edge_ent, "node_feat_size": r.node_feat_size, "node_feat_ent": r.node_feat_ent})
        for r in var_search_summary_df.itertuples(index=False)
    }
    var_stage_d_coeffs = build_stage_d_candidates(
        best_cfg=best_pre,
        grids={"edge_size": GRID_EDGE_SIZE, "edge_ent": GRID_EDGE_ENT, "node_feat_size": GRID_NODE_FEAT_SIZE, "node_feat_ent": GRID_NODE_FEAT_ENT},
        seen_cfgs=seen_cfgs,
        budget=VAR_STAGE_D_BUDGET,
    )
    if var_stage_d_coeffs:
        stage_d_epochs = VAR_EPOCHS_SEARCH if VAR_STAGE_D_USE_SEARCH_EPOCHS else VAR_EPOCHS_FINAL
        var_stage_d_seed_df, var_stage_d_mask_seed_df, var_stage_d_node_df, var_stage_d_summary = run_var_stage(
            stage_name="D",
            coeff_dicts=var_stage_d_coeffs,
            epochs=stage_d_epochs,
            seeds=VAR_SEEDS,
            target_nodes=var_search_target_nodes,
            base_model=var_base_model,
            graph_cache=var_graph_cache,
            pred_cache=var_pred_cache,
            dataset=var_dataset,
            train_graph_indices=var_train_graph_indices,
            node_types=var_node_types_eff,
            incident_edge_cache=var_incident_edge_cache,
            lr=VAR_LR,
        )
        var_seed_parts.append(var_stage_d_seed_df)
        var_mask_seed_parts.append(var_stage_d_mask_seed_df)
        var_node_parts.append(var_stage_d_node_df)
        var_summary_parts.append(var_stage_d_summary)
    else:
        var_stage_d_summary = pd.DataFrame()
else:
    var_stage_d_summary = pd.DataFrame()

var_stage_summary_search_df = pd.concat(var_summary_parts, ignore_index=True) if var_summary_parts else pd.DataFrame()
if var_stage_summary_search_df.empty:
    raise RuntimeError("Variability search stages produced no valid results.")

var_best_search_df = select_top_k_variability_configs(var_stage_summary_search_df, k=1)
if var_best_search_df.empty:
    raise RuntimeError("Variability search produced no finite mask variability scores.")

var_best_search_row = var_best_search_df.iloc[0]
var_best_config = {
    "edge_size": float(var_best_search_row["edge_size"]),
    "edge_ent": float(var_best_search_row["edge_ent"]),
    "node_feat_size": float(var_best_search_row["node_feat_size"]),
    "node_feat_ent": float(var_best_search_row["node_feat_ent"]),
    "lr": float(VAR_LR),
    "epochs": int(VAR_EPOCHS_FINAL),
}


c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\scripts\explainer\explainer_utils.py:114: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



Variability candidate nodes: 201
Variability search nodes: 30
Variability final nodes: 150
[A] start: 210 runs


Stage A:   0%|          | 0/210 [00:00<?, ?it/s]

[A] done: 210/210 runs in 11464.7s


VarMask Stage A:   0%|          | 0/210 [00:00<?, ?it/s]

...

VarMask Stage B:   0%|          | 0/210 [00:00<?, ?it/s]

[C] start: 45 runs


Stage C:   0%|          | 0/45 [00:00<?, ?it/s]

[C] done: 45/45 runs in 1986.6s


VarMask Stage C:   0%|          | 0/45 [00:00<?, ?it/s]

In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Run FINAL + build outputs
var_final_coeffs = [{
    "edge_size": var_best_config["edge_size"],
    "edge_ent": var_best_config["edge_ent"],
    "node_feat_size": var_best_config["node_feat_size"],
    "node_feat_ent": var_best_config["node_feat_ent"],
}]

var_final_seed_df, var_final_mask_seed_df, var_final_node_df, var_final_summary_df = run_var_stage(
    stage_name="FINAL",
    coeff_dicts=var_final_coeffs,
    epochs=VAR_EPOCHS_FINAL,
    seeds=VAR_SEEDS,
    target_nodes=var_final_target_nodes,
    base_model=var_base_model,
    graph_cache=var_graph_cache,
    pred_cache=var_pred_cache,
    dataset=var_dataset,
    train_graph_indices=var_train_graph_indices,
    node_types=var_node_types_eff,
    incident_edge_cache=var_incident_edge_cache,
    lr=VAR_LR,
)

var_seed_df = pd.concat(var_seed_parts + [var_final_seed_df], ignore_index=True)
var_mask_seed_df = pd.concat(var_mask_seed_parts + [var_final_mask_seed_df], ignore_index=True)
var_node_df = pd.concat(var_node_parts + [var_final_node_df], ignore_index=True)
var_stage_summary_df = pd.concat(var_summary_parts + [var_final_summary_df], ignore_index=True)

print("Variability best config (search -> final):")
print(var_best_config)
print("\nTop variability Stage A candidates:")
display(select_top_k_variability_configs(var_stage_a_summary, k=VAR_TOP_K_STAGE))
print("Top variability Stage B candidates:")
display(select_top_k_variability_configs(var_stage_b_summary, k=VAR_TOP_K_STAGE))
if not var_stage_c_summary.empty:
    print("Top variability Stage C candidates:")
    display(select_top_k_variability_configs(var_stage_c_summary, k=VAR_TOP_K_STAGE))
if not var_stage_d_summary.empty:
    print("Top variability Stage D candidates:")
    display(select_top_k_variability_configs(var_stage_d_summary, k=min(VAR_TOP_K_STAGE, max(1, VAR_STAGE_D_BUDGET))))
print("\nVariability FINAL summary:")
display(var_final_summary_df)


[FINAL] start: 5 runs


Stage FINAL:   0%|          | 0/5 [00:00<?, ?it/s]

[FINAL] done: 5/5 runs in 1942.8s


VarMask Stage FINAL:   0%|          | 0/5 [00:00<?, ?it/s]

Variability best config (search -> final):
{'edge_size': 0.1, 'edge_ent': 0.1, 'node_feat_size': 1e-12, 'node_feat_ent': 1e-12, 'lr': 0.01, 'epochs': 250}

Top variability Stage A candidates:


...

,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,...,fidelity_std,seed_runs,selection_sum,nodes_with_variability,mask_var_median,mask_var_iqr,mask_var_std,feature_mask_var_median,edge_mask_var_median,node_seed_runs_min
0,B,1.000000e-12,1.000000e-12,0.000100,0.01,0.01,150,0.052493,0.658892,0.046262,...,0.003307,5,18.5,30,0.044584,0.027075,0.022910,0.022013,0.066512,5.0
1,B,1.000000e-12,1.000000e-12,0.000010,0.01,0.01,150,0.052493,0.642782,0.045034,...,0.002222,5,19.0,30,0.044932,0.027161,0.022807,0.021722,0.066511,5.0
2,B,1.000000e-12,1.000000e-12,0.000001,0.01,0.01,150,0.052493,0.642782,0.045034,...,0.002222,5,19.0,30,0.044948,0.027161,0.022861,0.021721,0.066511,5.0


Top variability Stage C candidates:


,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,...,fidelity_std,seed_runs,selection_sum,nodes_with_variability,mask_var_median,mask_var_iqr,mask_var_std,feature_mask_var_median,edge_mask_var_median,node_seed_runs_min
0,C,0.1,0.10000,0.000001,0.01,0.01,150,0.030249,0.741883,0.023328,...,0.008116,5,13.5,30,0.020344,0.012068,0.013875,0.023154,0.012452,5.0
1,C,0.1,0.10000,0.000010,0.01,0.01,150,0.030249,0.741883,0.023328,...,0.008116,5,13.5,30,0.020345,0.012067,0.013870,0.023153,0.012452,5.0
2,C,0.1,0.00001,0.000001,0.01,0.01,150,0.029342,0.704967,0.023328,...,0.008098,5,13.5,30,0.020922,0.014613,0.013094,0.021385,0.014485,5.0



Variability FINAL summary:


,stage,edge_size,edge_ent,node_feat_size,node_feat_ent,lr,epochs,fidelity_median,fidelity_iqr,abs_delta_median,...,fidelity_std,seed_runs,selection_sum,nodes_with_variability,mask_var_median,mask_var_iqr,mask_var_std,feature_mask_var_median,edge_mask_var_median,node_seed_runs_min
0,FINAL,0.1,0.1,1.000000e-12,1.000000e-12,0.01,250,0.038923,0.993026,0.031221,...,0.002327,5,13.0,150,0.012222,0.018098,0.017679,0.015806,0.002847,5.0


In [ ]:
# VARIABILITY_GRIDSEARCH_SECTION_V1 - Export artifacts to results/experiments
if "var_context" not in globals() or var_context is None:
    raise RuntimeError("var_context missing. Run variability cells first.")
if "var_seed_df" not in globals() or not isinstance(var_seed_df, pd.DataFrame) or var_seed_df.empty:
    raise RuntimeError("var_seed_df missing/empty. Run variability cells first.")
if "var_node_df" not in globals() or not isinstance(var_node_df, pd.DataFrame):
    raise RuntimeError("var_node_df missing. Run variability cells first.")
if "var_stage_summary_df" not in globals() or not isinstance(var_stage_summary_df, pd.DataFrame) or var_stage_summary_df.empty:
    raise RuntimeError("var_stage_summary_df missing/empty. Run variability cells first.")

var_context.output_dir.mkdir(parents=True, exist_ok=True)
var_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

var_seed_csv = var_context.output_dir / f"gnn_variability_seed_level_{var_timestamp}.csv"
var_node_csv = var_context.output_dir / f"gnn_variability_node_level_{var_timestamp}.csv"
var_stage_csv = var_context.output_dir / f"gnn_variability_stage_summary_{var_timestamp}.csv"
var_best_json = var_context.output_dir / f"gnn_variability_best_config_{var_timestamp}.json"
var_run_json = var_context.output_dir / f"gnn_variability_run_config_{var_timestamp}.json"

var_seed_df.to_csv(var_seed_csv, index=False)
var_node_df.to_csv(var_node_csv, index=False)
var_stage_summary_df.to_csv(var_stage_csv, index=False)

var_best_payload = {
    "timestamp": var_timestamp,
    "objective": "minimize_mask_value_var_then_maximize_fidelity",
    "best_config": {
        "edge_size": float(var_best_config["edge_size"]),
        "edge_ent": float(var_best_config["edge_ent"]),
        "node_feat_size": float(var_best_config["node_feat_size"]),
        "node_feat_ent": float(var_best_config["node_feat_ent"]),
        "lr": float(var_best_config["lr"]),
        "epochs": int(var_best_config["epochs"]),
    },
}
with var_best_json.open("w", encoding="utf-8") as h:
    json.dump(var_best_payload, h, indent=2)

var_run_payload = {
    "timestamp": var_timestamp,
    "model_path": str(var_context.model_path),
    "data_path": str(var_context.data_path),
    "split_path": str(var_context.split_path),
    "graph_scope": str(GRAPH_SCOPE),
    "first_graph_per_component": bool(FIRST_GRAPH_PER_COMPONENT),
    "component_key": str(COMPONENT_KEY),
    "node_types": [str(nt) for nt in var_node_types_eff],
    "search_nodes": int(VAR_SEARCH_NODES),
    "final_nodes": int(VAR_FINAL_NODES),
    "seeds": [int(s) for s in VAR_SEEDS],
    "min_seed_runs_per_node": int(VAR_MIN_SEED_RUNS_PER_NODE),
    "sparsity_target": float(VAR_SPARSITY),
    "mask_scope": "feature+edge",
    "variability_metric": "mask_value_var",
    "ranking": ["mask_var_median_asc", "fidelity_median_desc", "selection_sum_asc", "runtime_sec_asc"],
    "mask_baseline_mode": str(MASK_BASELINE_MODE),
    "ig_baseline_mode": str(IG_BASELINE_MODE),
    "gnn_explanation_type": str(GNN_EXPLANATION_TYPE),
    "epochs_search": int(VAR_EPOCHS_SEARCH),
    "epochs_final": int(VAR_EPOCHS_FINAL),
    "lr": float(VAR_LR),
    "stage_d_budget": int(VAR_STAGE_D_BUDGET),
    "rows_seed_level": int(len(var_seed_df)),
    "rows_node_level": int(len(var_node_df)),
    "rows_stage_summary": int(len(var_stage_summary_df)),
    "artifacts": {
        "seed_level": str(var_seed_csv),
        "node_level": str(var_node_csv),
        "stage_summary": str(var_stage_csv),
        "best_config": str(var_best_json),
        "run_config": str(var_run_json),
    },
}
with var_run_json.open("w", encoding="utf-8") as h:
    json.dump(var_run_payload, h, indent=2)

assert np.isclose(float(VAR_SPARSITY), 0.8), "VAR_SPARSITY must be 0.8"
assert int(len(VAR_SEEDS)) == 5, "VAR_SEEDS must contain 5 seeds"
assert not var_stage_summary_df.empty, "var_stage_summary_df is empty"
assert np.isfinite(pd.to_numeric(var_stage_summary_df["mask_var_median"], errors="coerce")).any(), "No finite mask_var_median"
for p in [var_seed_csv, var_node_csv, var_stage_csv, var_best_json, var_run_json]:
    if not p.exists():
        raise RuntimeError(f"Missing artifact: {p}")

print("Saved variability artifacts:")
print(f"  seed level: {var_seed_csv}")
print(f"  node level: {var_node_csv}")
print(f"  stage summary: {var_stage_csv}")
print(f"  best config: {var_best_json}")
print(f"  run config: {var_run_json}")


Saved variability artifacts:
  seed level: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\results\experiments\gnn_variability_seed_level_20260308_090520.csv
  node level: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\results\experiments\gnn_variability_node_level_20260308_090520.csv
  stage summary: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\results\experiments\gnn_variability_stage_summary_20260308_090520.csv
  best config: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\results\experiments\gnn_variability_best_config_20260308_090520.json
  run config: c:\Uni\Lab\gnn4nmr-lab\gnn4nmr\results\experiments\gnn_variability_run_config_20260308_090520.json


## Integrated Gradients: Fidelity nach Baseline

Dieses Experiment vergleicht die IG-Fidelity (`fid_plus_model` und `fid_minus_model`) fuer verschiedene Baselines (z. B. `scientific`, `mean`, `zero`).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

# Konfiguration (bei Bedarf anpassen)
IG_BASELINE_CANDIDATES = ['scientific', 'mean', 'zero', 'random']
IG_BASELINE_MAX_GRAPHS = 15
IG_BASELINE_MAX_NODES_PER_GRAPH = 0  # 0 = alle Nodes im gewaehlten Graph
IG_BASELINE_SEED = 0

ig_baseline_node_frames = []
ig_baseline_errors = []

for baseline_mode in IG_BASELINE_CANDIDATES:
    print(f'>> Run baseline={baseline_mode}')
    try:
        node_df_i, _, _, _ = run_experiments_evaluation(
            node_types=tuple(node_types_widget.value),
            sparsity=float(sparsity_widget.value),
            mask_baseline_mode='match_ig_baseline',
            ig_baseline_mode=baseline_mode,
            gnn_epochs=int(gnn_epochs_widget.value),
            gnn_lr=float(gnn_lr_widget.value),
            gnn_explanation_type=gnn_explanation_type_widget.value,
            ig_n_steps=int(ig_n_steps_widget.value),
            graph_scope=graph_scope_widget.value,
            first_graph_per_component=bool(first_graph_per_component_widget.value),
            component_key=component_key_widget.value.strip() or 'compound',
            max_graphs=IG_BASELINE_MAX_GRAPHS,
            max_nodes_per_graph=int(IG_BASELINE_MAX_NODES_PER_GRAPH),
            include_gnn_edge_report=False,
            output_dir=output_dir_widget.value,
            seed=int(IG_BASELINE_SEED),
            verbose=False,
        )

        ig_df_i = node_df_i[node_df_i['method'] == 'integrated_gradients'].copy()
        ig_df_i['ig_baseline_mode'] = baseline_mode
        ig_baseline_node_frames.append(ig_df_i)

        median_fid_plus = pd.to_numeric(ig_df_i['fid_plus_model'], errors='coerce').median()
        median_fid_minus = pd.to_numeric(ig_df_i['fid_minus_model'], errors='coerce').median()
        print(
            f'   IG rows: {len(ig_df_i)} | '
            f'median(fid_plus_model)={median_fid_plus:.6f} | '
            f'median(fid_minus_model)={median_fid_minus:.6f}'
        )
    except Exception as exc:
        ig_baseline_errors.append({'ig_baseline_mode': baseline_mode, 'error': str(exc)})
        print(f'   Fehler fuer baseline={baseline_mode}: {exc}')

if not ig_baseline_node_frames:
    raise RuntimeError('Keine IG-Ergebnisse fuer Baseline-Vergleich erzeugt.')

ig_baseline_node_df = pd.concat(ig_baseline_node_frames, ignore_index=True)
ig_baseline_node_df['fid_plus_model'] = pd.to_numeric(ig_baseline_node_df['fid_plus_model'], errors='coerce')
ig_baseline_node_df['fid_minus_model'] = pd.to_numeric(ig_baseline_node_df['fid_minus_model'], errors='coerce')

ig_baseline_summary_df = (
    ig_baseline_node_df
    .groupby(['ig_baseline_mode', 'node_type'], dropna=False)
    .agg(
        n=('fid_plus_model', 'count'),
        fid_plus_median=('fid_plus_model', 'median'),
        fid_plus_mean=('fid_plus_model', 'mean'),
        fid_plus_std=('fid_plus_model', 'std'),
        fid_minus_median=('fid_minus_model', 'median'),
    )
    .reset_index()
    .sort_values(['node_type', 'fid_plus_median'], ascending=[True, False])
)

print('IG baseline summary (fidelity):')
display(ig_baseline_summary_df)

if ig_baseline_errors:
    print('Baselines mit Fehlern:')
    display(pd.DataFrame(ig_baseline_errors))

UNI_BLUE = '#004e9f'
UNI_YELLOW = '#fcba00'
node_type_colors = {'H': UNI_BLUE, 'C': UNI_YELLOW}

svg_output_dir = Path(output_dir_widget.value) / 'ig_baseline_svgs'
svg_output_dir.mkdir(parents=True, exist_ok=True)
svg_export_enabled = True
try:
    import kaleido  # noqa: F401
except Exception:
    svg_export_enabled = False
    print('Hinweis: SVG-Export deaktiviert (kaleido nicht verfuegbar).')

node_types_plot = [nt for nt in ['H', 'C'] if nt in set(ig_baseline_node_df['node_type'].astype(str))]
if not node_types_plot:
    node_types_plot = sorted(ig_baseline_node_df['node_type'].dropna().astype(str).unique().tolist())

tradeoff_by_type = {}

for node_type_sel in node_types_plot:
    node_df_plot = ig_baseline_node_df[ig_baseline_node_df['node_type'].astype(str) == node_type_sel].copy()
    summary_df_plot = ig_baseline_summary_df[ig_baseline_summary_df['node_type'].astype(str) == node_type_sel].copy()
    tradeoff_by_type[node_type_sel] = summary_df_plot.copy()

    fig_box = px.box(
        node_df_plot,
        x='ig_baseline_mode',
        y='fid_plus_model',
        points='outliers',
        category_orders={'ig_baseline_mode': IG_BASELINE_CANDIDATES},
        title=f'IG Fidelity (+) by Baseline - Node Type {node_type_sel}',
        labels={'ig_baseline_mode': 'Baseline (IG)', 'fid_plus_model': 'Fidelity (+)'},
    )
    fig_box.update_traces(marker_color=node_type_colors.get(node_type_sel, UNI_BLUE))
    fig_box.update_layout(template='plotly_white')
    fig_box.show()
    if svg_export_enabled:
        fig_box.write_image(str(svg_output_dir / f'ig_fidelity_plus_box_{node_type_sel}.svg'))

    fig_box_minus = px.box(
        node_df_plot,
        x='ig_baseline_mode',
        y='fid_minus_model',
        points='outliers',
        category_orders={'ig_baseline_mode': IG_BASELINE_CANDIDATES},
        title=f'IG Fidelity (-) by Baseline - Node Type {node_type_sel}',
        labels={'ig_baseline_mode': 'Baseline (IG)', 'fid_minus_model': 'Fidelity (-)'},
    )
    fig_box_minus.update_traces(marker_color=node_type_colors.get(node_type_sel, UNI_BLUE))
    fig_box_minus.update_layout(template='plotly_white')
    fig_box_minus.show()
    if svg_export_enabled:
        fig_box_minus.write_image(str(svg_output_dir / f'ig_fidelity_minus_box_{node_type_sel}.svg'))

    fig_median = px.bar(
        summary_df_plot,
        x='ig_baseline_mode',
        y='fid_plus_median',
        category_orders={'ig_baseline_mode': IG_BASELINE_CANDIDATES},
        title=f'Median Fidelity (+) by IG Baseline - Node Type {node_type_sel}',
        labels={'ig_baseline_mode': 'Baseline (IG)', 'fid_plus_median': 'Median Fidelity (+)'},
    )
    fig_median.update_traces(marker_color=node_type_colors.get(node_type_sel, UNI_BLUE))
    fig_median.update_layout(template='plotly_white')
    fig_median.show()
    if svg_export_enabled:
        fig_median.write_image(str(svg_output_dir / f'ig_fidelity_plus_median_{node_type_sel}.svg'))

    fig_median_minus = px.bar(
        summary_df_plot,
        x='ig_baseline_mode',
        y='fid_minus_median',
        category_orders={'ig_baseline_mode': IG_BASELINE_CANDIDATES},
        title=f'Median Fidelity (-) by IG Baseline - Node Type {node_type_sel}',
        labels={'ig_baseline_mode': 'Baseline (IG)', 'fid_minus_median': 'Median Fidelity (-)'},
    )
    fig_median_minus.update_traces(marker_color=node_type_colors.get(node_type_sel, UNI_BLUE))
    fig_median_minus.update_layout(template='plotly_white')
    fig_median_minus.show()
    if svg_export_enabled:
        fig_median_minus.write_image(str(svg_output_dir / f'ig_fidelity_minus_median_{node_type_sel}.svg'))


tradeoff_types = [nt for nt in ['H', 'C'] if nt in tradeoff_by_type and not tradeoff_by_type[nt].empty]
if not tradeoff_types:
    tradeoff_types = [nt for nt in node_types_plot if nt in tradeoff_by_type and not tradeoff_by_type[nt].empty]

if tradeoff_types:
    fig_tradeoff_pair = make_subplots(
        rows=1,
        cols=len(tradeoff_types),
        subplot_titles=[f'Node Type {nt}' for nt in tradeoff_types],
        horizontal_spacing=0.10,
    )

    for col_idx, node_type_sel in enumerate(tradeoff_types, start=1):
        trade_df = tradeoff_by_type[node_type_sel]
        fig_tradeoff_pair.add_trace(
            go.Scatter(
                x=trade_df['fid_plus_median'],
                y=trade_df['fid_minus_median'],
                mode='markers+text',
                text=trade_df['ig_baseline_mode'],
                textposition='top center',
                marker=dict(color=node_type_colors.get(node_type_sel, UNI_BLUE), size=11),
                name=f'Node Type {node_type_sel}',
                showlegend=False,
            ),
            row=1,
            col=col_idx,
        )
        fig_tradeoff_pair.update_xaxes(title_text='Median Fidelity (+)', row=1, col=col_idx)
        fig_tradeoff_pair.update_yaxes(title_text='Median Fidelity (-)', row=1, col=col_idx)

    fig_tradeoff_pair.update_layout(
        template='plotly_white',
        title='Fidelity (+) vs Fidelity (-) by Node Type',
    )
    fig_tradeoff_pair.show()
    if svg_export_enabled:
        fig_tradeoff_pair.write_image(str(svg_output_dir / 'ig_tradeoff_scatter_H_C.svg'))

if svg_export_enabled:
    print(f'SVGs gespeichert unter: {svg_output_dir}')


>> Run baseline=scientific


Graph evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

   IG rows: 14 | median(fid_plus_model)=0.043721 | median(fid_minus_model)=0.017349
IG baseline summary (fidelity):


,ig_baseline_mode,node_type,n,fid_plus_median,fid_plus_mean,fid_plus_std,fid_minus_median
0,scientific,C,6,2.489564,3.492718,1.925035,0.162626
1,scientific,H,8,0.012651,0.017855,0.020422,0.004994


Hinweis: SVG-Export deaktiviert (kaleido nicht verfuegbar).


### IG Baseline Fidelity (for different Sparsity)

Trade-off-Scatter fuer mehrere Sparsities: pro Baseline werden `fid_plus_median` und `fid_minus_median` ueber den Sparsity-Grid dargestellt (H und C nebeneinander).


In [ ]:
# Sparsity-Sweep Konfiguration
SPARSITY_GRID = [0.70, 0.80, 0.90]
BASELINE_GRID = list(IG_BASELINE_CANDIDATES)
SWEEP_MAX_GRAPHS = 0  # 0 oder None = alle gefilterten Graphen

sweep_rows = []
sweep_errors = []

for baseline_mode in BASELINE_GRID:
    for sparsity_val in SPARSITY_GRID:
        print(f'>> sweep baseline={baseline_mode} sparsity={sparsity_val:.2f}')
        try:
            node_df_s, _, _, _ = run_experiments_evaluation(
                node_types=tuple(node_types_widget.value),
                sparsity=float(sparsity_val),
                mask_baseline_mode='match_ig_baseline',
                ig_baseline_mode=baseline_mode,
                gnn_epochs=int(gnn_epochs_widget.value),
                gnn_lr=float(gnn_lr_widget.value),
                gnn_explanation_type=gnn_explanation_type_widget.value,
                ig_n_steps=int(ig_n_steps_widget.value),
                graph_scope=graph_scope_widget.value,
                first_graph_per_component=bool(first_graph_per_component_widget.value),
                component_key=component_key_widget.value.strip() or 'compound',
                max_graphs=(None if SWEEP_MAX_GRAPHS in (0, None) else int(SWEEP_MAX_GRAPHS)),
                max_nodes_per_graph=int(IG_BASELINE_MAX_NODES_PER_GRAPH),
                include_gnn_edge_report=False,
                output_dir=output_dir_widget.value,
                seed=int(IG_BASELINE_SEED),
                verbose=False,
            )

            ig_df_s = node_df_s[node_df_s['method'] == 'integrated_gradients'].copy()
            if ig_df_s.empty:
                continue
            ig_df_s['ig_baseline_mode'] = str(baseline_mode)
            ig_df_s['sparsity'] = float(sparsity_val)
            sweep_rows.append(ig_df_s)
        except Exception as exc:
            sweep_errors.append({
                'ig_baseline_mode': str(baseline_mode),
                'sparsity': float(sparsity_val),
                'error': str(exc),
            })
            print(f'   error: {exc}')

if not sweep_rows:
    raise RuntimeError('No results generated for sparsity sweep.')

sweep_node_df = pd.concat(sweep_rows, ignore_index=True)
sweep_node_df['fid_plus_model'] = pd.to_numeric(sweep_node_df['fid_plus_model'], errors='coerce')
sweep_node_df['fid_minus_model'] = pd.to_numeric(sweep_node_df['fid_minus_model'], errors='coerce')

sweep_summary_df = (
    sweep_node_df
    .groupby(['ig_baseline_mode', 'sparsity', 'node_type'], dropna=False)
    .agg(
        n=('fid_plus_model', 'count'),
        fid_plus_median=('fid_plus_model', 'median'),
        fid_minus_median=('fid_minus_model', 'median'),
    )
    .reset_index()
)

print('Sparsity sweep summary:')
display(sweep_summary_df)
if sweep_errors:
    print('Sweep runs with errors:')
    display(pd.DataFrame(sweep_errors))

node_types_trade = [nt for nt in ['H', 'C'] if nt in set(sweep_summary_df['node_type'].astype(str))]
if not node_types_trade:
    node_types_trade = sorted(sweep_summary_df['node_type'].dropna().astype(str).unique().tolist())

if node_types_trade:
    fig_trade_sweep = make_subplots(
        rows=1,
        cols=len(node_types_trade),
        subplot_titles=[f'Node Type {nt}' for nt in node_types_trade],
        horizontal_spacing=0.10,
    )

    baseline_colors = {
        'scientific': '#004e9f',
        'mean': '#fcba00',
        'zero': '#909085',
        'random': '#c0392b',
        'min': '#1a7a4a',
        'max': '#1a1a1a',
    }

    for col_idx, node_type_sel in enumerate(node_types_trade, start=1):
        sub = sweep_summary_df[sweep_summary_df['node_type'].astype(str) == node_type_sel].copy()
        for baseline_mode in BASELINE_GRID:
            bdf = sub[sub['ig_baseline_mode'].astype(str) == str(baseline_mode)].copy()
            if bdf.empty:
                continue
            bdf = bdf.sort_values('sparsity')
            fig_trade_sweep.add_trace(
                go.Scatter(
                    x=bdf['fid_plus_median'],
                    y=bdf['fid_minus_median'],
                    mode='markers+lines+text',
                    text=[f"s={s:.2f}" for s in bdf['sparsity']],
                    textposition='top center',
                    marker=dict(
                        size=[8 + 16 * float(s) for s in bdf['sparsity']],
                        color=baseline_colors.get(str(baseline_mode), '#1a1a1a'),
                    ),
                    line=dict(color=baseline_colors.get(str(baseline_mode), '#1a1a1a'), width=2),
                    name=str(baseline_mode),
                    legendgroup=str(baseline_mode),
                    showlegend=(col_idx == 1),
                ),
                row=1,
                col=col_idx,
            )
        fig_trade_sweep.update_xaxes(title_text='Median Fidelity (+)', row=1, col=col_idx)
        fig_trade_sweep.update_yaxes(title_text='Median Fidelity (-)', row=1, col=col_idx)

    fig_trade_sweep.update_layout(
        template='plotly_white',
        title='IG Trade-off Scatter over Sparsity (per Baseline)',
        legend_title_text='IG Baseline',
    )
    fig_trade_sweep.show()

    if 'svg_export_enabled' in globals() and svg_export_enabled:
        fig_trade_sweep.write_image(str(svg_output_dir / 'ig_tradeoff_scatter_sparsity_H_C.svg'))
        print(f'SVG saved: {svg_output_dir / "ig_tradeoff_scatter_sparsity_H_C.svg"}')


>> sweep baseline=scientific sparsity=0.70


Graph evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

>> sweep baseline=scientific sparsity=0.80


Graph evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

>> sweep baseline=scientific sparsity=0.90


Graph evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

Sparsity sweep summary:


,ig_baseline_mode,sparsity,node_type,n,fid_plus_median,fid_minus_median
0,scientific,0.7,C,6,2.736925,0.106947
1,scientific,0.7,H,8,0.014031,0.001080
2,scientific,0.8,C,6,2.840694,0.173415
3,scientific,0.8,H,8,0.014414,0.002506
4,scientific,0.9,C,6,2.489564,0.162626
5,scientific,0.9,H,8,0.012651,0.004994
